# ClickHouse User-Level Feature Engineering
## Objective
Create user-level features for fraud detection using ClickHouse
- **Target Users**: Users who transacted on 2025-07-01 (July 2025 fraud data)
- **Time Windows**: 3d and 7d lookback
- **Lookback Period**: 7 days before cutoff date
- **Note**: Features exclude same-day data to prevent data leakage
- **Data Source**: `stixor_mbar_v` (includes MBAR account type columns)

## Cutoff Date Architecture
All feature tables include a `cutoff_date` column that enables:
1. **Multiple Time Periods**: Run the script for different dates to build historical feature sets
2. **Easy Filtering**: When creating combined features, filter by cutoff_date to ensure consistency
3. **Time-Series Analysis**: Compare features across different cutoff dates
4. **Model Training**: Use cutoff_date to create training/validation/test splits

**Usage**: Change the `CUTOFF_DATE` variable at the top to process features for different dates.

In [19]:
# ClickHouse Connection Setup
from clickhouse_driver import Client
import pandas as pd
from datetime import datetime

# ===== GLOBAL CONFIGURATION =====
# This cutoff date will be used across all feature tables
# Change this date to process features for different time periods
CUTOFF_DATE = '2025-06-05'  # Changed to match MBAR data range (July 2025)

# Source table configuration
SOURCE_TABLE = 'stixor_mbar_v'  # Table with MBAR columns
# Alternative: 'stixor_iar_distributed' (without MBAR)

print(f"🎯 Global Cutoff Date: {CUTOFF_DATE}")
print(f"📊 Source Table: {SOURCE_TABLE}")
print("=" * 80)

# ClickHouse configuration
CLICKHOUSE_CONFIG = {
    'host': '10.205.161.118',
    'port': 9000,  # Native TCP port for ClickHouse
    'database': 'public',
    'user': 'default',
    'password': 'DfsTeChB1'  # Update if password is set
}

print("🔌 Connecting to ClickHouse...")
print(f"📍 Host: {CLICKHOUSE_CONFIG['host']}:{CLICKHOUSE_CONFIG['port']}")
print(f"🗄️  Database: {CLICKHOUSE_CONFIG['database']}")

# Create ClickHouse client
try:
    clickhouse_client = Client(
        host=CLICKHOUSE_CONFIG['host'],
        port=CLICKHOUSE_CONFIG['port'],
        database=CLICKHOUSE_CONFIG['database'],
        user=CLICKHOUSE_CONFIG['user'],
        password=CLICKHOUSE_CONFIG['password'],
        settings={
            'max_execution_time': 3600,  # 1 hour timeout
            'send_timeout': 300,
            'receive_timeout': 300,
            'connect_timeout': 10
        }
    )
    
    # Test connection
    result = clickhouse_client.execute('SELECT version()')
    clickhouse_version = result[0][0]
    
    print(f"✅ ClickHouse connection successful!")
    print(f"📦 ClickHouse Version: {clickhouse_version}")
    
    # Show available databases
    databases = clickhouse_client.execute('SHOW DATABASES')
    print(f"🗂️  Available Databases: {[db[0] for db in databases]}")
    
    # Show tables in current database
    tables = clickhouse_client.execute(f'SHOW TABLES FROM {CLICKHOUSE_CONFIG["database"]}')
    if tables:
        print(f"📊 Tables in '{CLICKHOUSE_CONFIG['database']}': {[tbl[0] for tbl in tables]}")
    else:
        print(f"📊 No tables found in '{CLICKHOUSE_CONFIG['database']}'")
    
except Exception as e:
    print(f"❌ Failed to connect to ClickHouse: {str(e)}")
    print("💡 Tips:")
    print("   • Ensure ClickHouse server is running")
    print("   • Check if port 9000 is accessible")
    print("   • Verify credentials and permissions")
    raise

🎯 Global Cutoff Date: 2025-06-05
📊 Source Table: stixor_mbar_v
🔌 Connecting to ClickHouse...
📍 Host: 10.205.161.118:9000
🗄️  Database: public
✅ ClickHouse connection successful!
📦 ClickHouse Version: 25.10.1.3796
🗂️  Available Databases: ['INFORMATION_SCHEMA', 'default', 'information_schema', 'public', 'system']
📊 Tables in 'public': ['ac_from_features', 'ac_from_features_distributed', 'ac_from_features_local', 'combined_features_balanced_1_10_distributed', 'combined_features_balanced_1_10_local', 'combined_features_balanced_1_1_distributed', 'combined_features_balanced_1_1_local', 'combined_features_balanced_1_20_distributed', 'combined_features_balanced_1_20_local', 'combined_features_balanced_1_5_distributed', 'combined_features_balanced_1_5_local', 'combined_features_distributed', 'combined_features_local', 'fraud', 'fraud_distributed', 'stixor_iar', 'stixor_iar_distributed', 'stixor_locations', 'stixor_locations_distributed', 'stixor_mbar_v', 'stixor_mbar_v_distributed', 'transact

## Helper Functions
Define reusable functions for executing queries and fetching data

In [6]:
def execute_clickhouse_query(query, return_df=False):
    """
    Execute a ClickHouse query and optionally return results as DataFrame
    
    Args:
        query: SQL query string
        return_df: If True, return results as pandas DataFrame
    
    Returns:
        Query results or None
    """
    try:
        print(f"🔄 Executing query...")
        result = clickhouse_client.execute(query, with_column_types=True)
        
        if return_df and result:
            # Extract data and column info
            data = result[0] if isinstance(result, tuple) else result
            
            if isinstance(result, tuple) and len(result) > 1:
                # Has column type information
                columns = [col[0] for col in result[1]]
                df = pd.DataFrame(data, columns=columns)
            else:
                df = pd.DataFrame(data)
            
            print(f"✅ Query executed successfully! Rows: {len(df):,}")
            return df
        else:
            print(f"✅ Query executed successfully!")
            return result
            
    except Exception as e:
        print(f"❌ Query execution failed: {str(e)}")
        raise


def show_table_info(table_name):
    """Display table schema and row count"""
    print(f"\n📋 Table Info: {table_name}")
    print("=" * 80)
    
    # Get schema
    schema_query = f"DESCRIBE TABLE {table_name}"
    schema = clickhouse_client.execute(schema_query)
    
    print("\n🔧 Schema:")
    for col in schema[:10]:  # Show first 10 columns
        print(f"   {col[0]:30s} {col[1]}")
    if len(schema) > 10:
        print(f"   ... and {len(schema) - 10} more columns")
    
    # Get row count
    count_query = f"SELECT count(*) FROM {table_name}"
    count = clickhouse_client.execute(count_query)[0][0]
    print(f"\n📊 Total Rows: {count:,}")
    
    return count

## Step 1: Create Feature Tables
Create local and distributed tables for user-level features

In [4]:
# Drop existing tables if they exist
drop_queries = [
    "DROP TABLE IF EXISTS public.ac_from_features_distributed ON CLUSTER my_cluster_2shards",
    "DROP TABLE IF EXISTS public.ac_from_features_local ON CLUSTER my_cluster_2shards"
]

print("🗑️  Dropping existing tables...")
for query in drop_queries:
    try:
        clickhouse_client.execute(query)
        print(f"✅ Executed: {query.split('DROP TABLE IF EXISTS')[1].split('ON')[0].strip()}")
    except Exception as e:
        print(f"⚠️  Warning: {str(e)}")

print("\n✅ Cleanup complete!")

🗑️  Dropping existing tables...
✅ Executed: public.ac_from_features_distributed
✅ Executed: public.ac_from_features_local

✅ Cleanup complete!


In [5]:
# Create local table for user features
create_local_table_query = """
CREATE TABLE IF NOT EXISTS public.ac_from_features_local
ON CLUSTER my_cluster_2shards
(
    ac_from String,
    cutoff_date Date DEFAULT '2025-05-31',
    
    -- 3 day aggregate features  
    total_txns_3d UInt32 DEFAULT 0,
    total_amount_3d Float64 DEFAULT 0,
    avg_amount_3d Float64 DEFAULT 0,
    median_amount_3d Float64 DEFAULT 0,
    max_amount_3d Float64 DEFAULT 0,
    min_amount_3d Float64 DEFAULT 0,
    unique_recipients_3d UInt32 DEFAULT 0,
    unique_channels_3d UInt32 DEFAULT 0,
    unique_types_3d UInt32 DEFAULT 0,
    
    -- 7 day aggregate features
    total_txns_7d UInt32 DEFAULT 0,
    total_amount_7d Float64 DEFAULT 0,
    avg_amount_7d Float64 DEFAULT 0,
    median_amount_7d Float64 DEFAULT 0,
    max_amount_7d Float64 DEFAULT 0,
    min_amount_7d Float64 DEFAULT 0,
    unique_recipients_7d UInt32 DEFAULT 0,
    unique_channels_7d UInt32 DEFAULT 0,
    unique_types_7d UInt32 DEFAULT 0,
    
    -- Channel preference features (7-day)
    most_used_channel_7d String DEFAULT '',
    last_used_channel String DEFAULT '',
    channel_diversity_score_7d Float64 DEFAULT 0,
    
    -- Transaction type features (7-day)
    most_used_type_7d String DEFAULT '',
    last_used_type String DEFAULT '',
    type_diversity_score_7d Float64 DEFAULT 0,
    
    -- Time-based features (7-day)
    night_txns_7d UInt32 DEFAULT 0,
    weekend_txns_7d UInt32 DEFAULT 0,
    peak_hour_txns_7d UInt32 DEFAULT 0,
    off_peak_hour_txns_7d UInt32 DEFAULT 0,
    
    -- Balance features (7-day)
    avg_start_balance_7d Float64 DEFAULT 0,
    avg_end_balance_7d Float64 DEFAULT 0,
    min_balance_7d Float64 DEFAULT 0,
    max_balance_7d Float64 DEFAULT 0,
    balance_volatility_7d Float64 DEFAULT 0,
    
    -- Recipient features (7-day)
    top_recipient_7d String DEFAULT '',
    avg_amount_per_recipient_7d Float64 DEFAULT 0,
    max_amount_to_single_recipient_7d Float64 DEFAULT 0,
    recipient_concentration_ratio_7d Float64 DEFAULT 0,
    
    -- Behavioral features
    avg_time_between_txns_7d Float64 DEFAULT 0,
    txn_frequency_score_7d Float64 DEFAULT 0,
    first_txn_time DateTime DEFAULT toDateTime('1970-01-01 00:00:00'),
    last_txn_time DateTime DEFAULT toDateTime('1970-01-01 00:00:00'),
    days_since_last_txn UInt32 DEFAULT 0,
    
    processing_timestamp DateTime DEFAULT now(),
    created_at Date DEFAULT today()
)
ENGINE = MergeTree()
ORDER BY ac_from
PARTITION BY toYYYYMM(processing_timestamp)
"""

print("📊 Creating local table: public.ac_from_features_local...")
clickhouse_client.execute(create_local_table_query)
print("✅ Local table created successfully!")
print("   • Table: public.ac_from_features_local")
print("   • Engine: MergeTree")
print("   • Order By: ac_from")
print("   • Partition By: toYYYYMM(processing_timestamp)")
print("   • Features: 38 feature columns + 2 metadata columns")

📊 Creating local table: public.ac_from_features_local...
✅ Local table created successfully!
   • Table: public.ac_from_features_local
   • Engine: MergeTree
   • Order By: ac_from
   • Partition By: toYYYYMM(processing_timestamp)
   • Features: 38 feature columns + 2 metadata columns


In [6]:
# Create distributed table
create_distributed_table_query = """
CREATE TABLE IF NOT EXISTS public.ac_from_features_distributed AS public.ac_from_features_local
ENGINE = Distributed(my_cluster_2shards, public, ac_from_features_local, cityHash64(ac_from))
"""

print("\n🌐 Creating distributed table: public.ac_from_features_distributed...")
clickhouse_client.execute(create_distributed_table_query)
print("✅ Distributed table created successfully!")
print("   • Table: public.ac_from_features_distributed")
print("   • Engine: Distributed")
print("   • Cluster: my_cluster_2shards")
print("   • Sharding Key: cityHash64(ac_from)")
print("   • Local Table: public.ac_from_features_local")

print("\n" + "="*80)
print("🎉 Table creation complete!")
print("="*80)


🌐 Creating distributed table: public.ac_from_features_distributed...
✅ Distributed table created successfully!
   • Table: public.ac_from_features_distributed
   • Engine: Distributed
   • Cluster: my_cluster_2shards
   • Sharding Key: cityHash64(ac_from)
   • Local Table: public.ac_from_features_local

🎉 Table creation complete!


In [7]:
# Verify tables were created
print("\n🔍 Verifying table creation...")

# Check local table
local_table_info = show_table_info("public.ac_from_features_local")

# Check distributed table  
print("\n" + "="*80)
distributed_query = "SELECT count(*) FROM public.ac_from_features_distributed"
dist_count = clickhouse_client.execute(distributed_query)[0][0]
print(f"📊 Distributed table row count: {dist_count:,}")

print("\n✅ All tables verified successfully!")


🔍 Verifying table creation...

📋 Table Info: public.ac_from_features_local

🔧 Schema:
   ac_from                        String
   cutoff_date                    Date
   total_txns_3d                  UInt32
   total_amount_3d                Float64
   avg_amount_3d                  Float64
   median_amount_3d               Float64
   max_amount_3d                  Float64
   min_amount_3d                  Float64
   unique_recipients_3d           UInt32
   unique_channels_3d             UInt32
   ... and 36 more columns

📊 Total Rows: 0

📊 Distributed table row count: 0

✅ All tables verified successfully!


## Step 2: Feature Engineering Query
Generate user-level features for active users on the cutoff date

**Important**: All user features will include the `cutoff_date` column for easy filtering when joining with transaction features.

In [27]:
# Configuration parameters
# CUTOFF_DATE is now set globally at the top of the notebook
LOOKBACK_DAYS = 7
CUTOFF_DATE='2025-06-15'
# Calculate start date
from datetime import datetime, timedelta
cutoff = datetime.strptime(CUTOFF_DATE, '%Y-%m-%d')
start_date = (cutoff - timedelta(days=LOOKBACK_DAYS - 1)).strftime('%Y-%m-%d')

print("📅 Feature Engineering Configuration:")
print(f"   • Cutoff Date: {CUTOFF_DATE} (Global Configuration)")
print(f"   • Lookback Days: {LOOKBACK_DAYS}")
print(f"   • Start Date: {start_date}")
print(f"   • Date Range: {start_date} to {CUTOFF_DATE}")
print(f"   • Time Windows: 1d, 3d, 7d")
print(f"\n💡 Note: cutoff_date column will be stored in all feature tables for filtering")


📅 Feature Engineering Configuration:
   • Cutoff Date: 2025-06-15 (Global Configuration)
   • Lookback Days: 7
   • Start Date: 2025-06-09
   • Date Range: 2025-06-09 to 2025-06-15
   • Time Windows: 1d, 3d, 7d

💡 Note: cutoff_date column will be stored in all feature tables for filtering


In [28]:
# Build parameterized feature engineering query (SELECT only for preview)
feature_query = f"""
INSERT INTO public.ac_from_features_distributed
WITH 
-- Get users who transacted on cutoff date
active_users AS (
    SELECT DISTINCT ac_from
    FROM public.stixor_iar_distributed
    WHERE data_date = toDate('{CUTOFF_DATE}')
      AND ac_from != ''),
-- Pre-calculate top channels and types per user ({LOOKBACK_DAYS}-day window)
user_channel_stats AS (
    SELECT 
        ac_from,
        trx_channel,
        count() as channel_count,
        row_number() OVER (PARTITION BY ac_from ORDER BY count() DESC) as channel_rank
    FROM (
        SELECT 
            ac_from,
            trx_channel
        FROM public.stixor_iar_distributed
        WHERE data_date >= toDate('{start_date}')
          AND data_date <= toDate('{CUTOFF_DATE}')
          AND ac_from GLOBAL IN (SELECT ac_from FROM active_users)
    )
    GROUP BY ac_from, trx_channel
),
user_type_stats AS (
    SELECT 
        ac_from,
        trx_type,
        count() as type_count,
        row_number() OVER (PARTITION BY ac_from ORDER BY count() DESC) as type_rank
    FROM (
        SELECT 
            ac_from,
            trx_type
        FROM public.stixor_iar_distributed
        WHERE data_date >= toDate('{start_date}')
          AND data_date <= toDate('{CUTOFF_DATE}')
          AND ac_from GLOBAL IN (SELECT ac_from FROM active_users)
    )
    GROUP BY ac_from, trx_type
),
user_recipient_stats AS (
    SELECT 
        ac_from,
        ac_to,
        count() as recipient_count,
        sum(start_balance) as total_to_recipient,
        row_number() OVER (PARTITION BY ac_from ORDER BY count() DESC) as recipient_rank
    FROM (
        SELECT 
            ac_from,
            ac_to,
            start_balance
        FROM public.stixor_iar_distributed
        WHERE data_date >= toDate('{start_date}')
          AND data_date <= toDate('{CUTOFF_DATE}')
          AND ac_from GLOBAL IN (SELECT ac_from FROM active_users)
          AND ac_to != ''
    )
    GROUP BY ac_from, ac_to
)

SELECT 
    main.ac_from,
    toDate('{CUTOFF_DATE}') as cutoff_date,
    
    -- 3-day features (excludes cutoff date, uses days_back 1-3)
    sumIf(1, main.days_back BETWEEN 1 AND 3) as total_txns_3d,
    sumIf(main.start_balance, main.days_back BETWEEN 1 AND 3) as total_amount_3d,
    avgIf(main.start_balance, main.days_back BETWEEN 1 AND 3) as avg_amount_3d,
    quantileIf(0.5)(main.start_balance, main.days_back BETWEEN 1 AND 3) as median_amount_3d,
    maxIf(main.start_balance, main.days_back BETWEEN 1 AND 3) as max_amount_3d,
    minIf(main.start_balance, main.days_back BETWEEN 1 AND 3) as min_amount_3d,
    uniqIf(main.ac_to, main.days_back BETWEEN 1 AND 3) as unique_recipients_3d,
    uniqIf(main.trx_channel, main.days_back BETWEEN 1 AND 3) as unique_channels_3d,
    uniqIf(main.trx_type, main.days_back BETWEEN 1 AND 3) as unique_types_3d,
    
    -- 7-day features (excludes cutoff date, uses days_back 1-7)
    sumIf(1, main.days_back BETWEEN 1 AND 7) as total_txns_7d,
    sumIf(main.start_balance, main.days_back BETWEEN 1 AND 7) as total_amount_7d,
    avgIf(main.start_balance, main.days_back BETWEEN 1 AND 7) as avg_amount_7d,
    quantileIf(0.5)(main.start_balance, main.days_back BETWEEN 1 AND 7) as median_amount_7d,
    maxIf(main.start_balance, main.days_back BETWEEN 1 AND 7) as max_amount_7d,
    minIf(main.start_balance, main.days_back BETWEEN 1 AND 7) as min_amount_7d,
    uniqIf(main.ac_to, main.days_back BETWEEN 1 AND 7) as unique_recipients_7d,
    uniqIf(main.trx_channel, main.days_back BETWEEN 1 AND 7) as unique_channels_7d,
    uniqIf(main.trx_type, main.days_back BETWEEN 1 AND 7) as unique_types_7d,
    
    -- Channel features (7-day for most_used, last_used, diversity)
    anyIf(ch.trx_channel, ch.channel_rank = 1) as most_used_channel_7d,
    argMax(main.trx_channel, main.trans_initiate_time) as last_used_channel,
    if(uniq(main.trx_channel) > 1, 
       1 - (max(ch.channel_count) / sum(ch.channel_count)), 0) as channel_diversity_score_7d,
    
    -- Type features (7-day for most_used, last_used, diversity)
    anyIf(ty.trx_type, ty.type_rank = 1) as most_used_type_7d,
    argMax(main.trx_type, main.trans_initiate_time) as last_used_type,
    if(uniq(main.trx_type) > 1, 
       1 - (max(ty.type_count) / sum(ty.type_count)), 0) as type_diversity_score_7d,
    
    -- Time-based features (7-day, excluding cutoff date)
    sumIf(1, toHour(main.trans_initiate_time) IN (2,3,4,5,6) AND main.days_back BETWEEN 1 AND 7) as night_txns_7d,
    sumIf(1, toDayOfWeek(main.trans_initiate_time) IN (6,7) AND main.days_back BETWEEN 1 AND 7) as weekend_txns_7d,
    sumIf(1, toHour(main.trans_initiate_time) BETWEEN 9 AND 17 AND main.days_back BETWEEN 1 AND 7) as peak_hour_txns_7d,
    sumIf(1, toHour(main.trans_initiate_time) NOT BETWEEN 9 AND 17 AND main.days_back BETWEEN 1 AND 7) as off_peak_hour_txns_7d,
    
    -- Balance features (7-day, excluding cutoff date)
    avgIf(main.start_balance, main.days_back BETWEEN 1 AND 7) as avg_start_balance_7d,
    avgIf(main.end_balance, main.days_back BETWEEN 1 AND 7) as avg_end_balance_7d,
    minIf(least(main.start_balance, main.end_balance), main.days_back BETWEEN 1 AND 7) as min_balance_7d,
    maxIf(greatest(main.start_balance, main.end_balance), main.days_back BETWEEN 1 AND 7) as max_balance_7d,
    stddevPopIf(main.start_balance, main.days_back BETWEEN 1 AND 7) as balance_volatility_7d,
    
    -- Recipient features (7-day, excluding cutoff date)
    anyIf(rs.ac_to, rs.recipient_rank = 1) as top_recipient_7d,
    avgIf(main.start_balance, main.days_back BETWEEN 1 AND 7 AND main.ac_to != '') as avg_amount_per_recipient_7d,
    maxIf(main.start_balance, main.days_back BETWEEN 1 AND 7 AND main.ac_to != '') as max_amount_to_single_recipient_7d,
    if(count(main.ac_from) > 0, max(rs.recipient_count) / count(main.ac_from), 0) as recipient_concentration_ratio_7d,
    
    -- Behavioral features (7-day)
    if(count(main.ac_from) > 1,
       dateDiff('hour', min(main.trans_initiate_time), max(main.trans_initiate_time)) / (count(main.ac_from) - 1),
       0) as avg_time_between_txns_7d,
    count(main.ac_from) / greatest(dateDiff('day', min(main.data_date), max(main.data_date)) + 1, 1) as txn_frequency_score_7d,
    min(main.trans_initiate_time) as first_txn_time,
    max(main.trans_initiate_time) as last_txn_time,
    dateDiff('day', max(main.data_date), toDate('{CUTOFF_DATE}')) as days_since_last_txn,
    
    now() as processing_timestamp,
    toDate(now()) as created_at

FROM (
    SELECT 
        ac_from,
        ac_to,
        trans_id,
        start_balance,
        end_balance,
        trx_channel,
        trx_type,
        trans_initiate_time,
        data_date,
        dateDiff('day', data_date, toDate('{CUTOFF_DATE}')) as days_back
    FROM public.stixor_iar_distributed
    WHERE data_date >= toDate('{start_date}')
      AND data_date <= toDate('{CUTOFF_DATE}')
      AND ac_from GLOBAL IN (SELECT ac_from FROM active_users)
    ORDER BY ac_from, trans_initiate_time
) main
GLOBAL LEFT JOIN user_channel_stats ch ON main.ac_from = ch.ac_from AND main.trx_channel = ch.trx_channel
GLOBAL LEFT JOIN user_type_stats ty ON main.ac_from = ty.ac_from AND main.trx_type = ty.trx_type  
GLOBAL LEFT JOIN user_recipient_stats rs ON main.ac_from = rs.ac_from AND main.ac_to = rs.ac_to
GROUP BY main.ac_from
"""

print("\n📝 Feature engineering query prepared")
print(f"   • Query length: {len(feature_query):,} characters")
print(f"   • Target table: public.ac_from_features_distributed")
print(f"   • Date range: {start_date} to {CUTOFF_DATE}")
print(f"   • Features: 3d window (9 features) + 7d window (27 features) = 36 total features")


📝 Feature engineering query prepared
   • Query length: 7,273 characters
   • Target table: public.ac_from_features_distributed
   • Date range: 2025-06-09 to 2025-06-15
   • Features: 3d window (9 features) + 7d window (27 features) = 36 total features


In [29]:
# Execute feature engineering query
print("\n" + "="*80)
print("🚀 Executing Feature Engineering Query...")
print("="*80)
print("\n⏳ This may take several minutes depending on data volume...")
print("   Please wait...")

import time
start_time = time.time()

try:
    # Execute the INSERT query
    clickhouse_client.execute(feature_query)
    
    elapsed_time = time.time() - start_time
    
    print(f"\n✅ Feature engineering completed successfully!")
    print(f"⏱️  Execution time: {elapsed_time:.2f} seconds ({elapsed_time/60:.2f} minutes)")
    
    # Get count of inserted rows
    count_query = "SELECT count(*) FROM public.ac_from_features_distributed"
    total_users = clickhouse_client.execute(count_query)[0][0]
    
    print(f"\n📊 Results:")
    print(f"   • Total users processed: {total_users:,}")
    print(f"   • Features per user: 36 (9 x 3d + 27 x 7d)")
    print(f"   • Total feature values: {total_users * 36:,}")
    
except Exception as e:
    elapsed_time = time.time() - start_time
    print(f"\n❌ Feature engineering failed after {elapsed_time:.2f} seconds")
    print(f"   Error: {str(e)}")
    raise


🚀 Executing Feature Engineering Query...

⏳ This may take several minutes depending on data volume...
   Please wait...

✅ Feature engineering completed successfully!
⏱️  Execution time: 33.48 seconds (0.56 minutes)

📊 Results:
   • Total users processed: 17,827,060
   • Features per user: 36 (9 x 3d + 27 x 7d)
   • Total feature values: 641,774,160

✅ Feature engineering completed successfully!
⏱️  Execution time: 33.48 seconds (0.56 minutes)

📊 Results:
   • Total users processed: 17,827,060
   • Features per user: 36 (9 x 3d + 27 x 7d)
   • Total feature values: 641,774,160


In [31]:
# Check for duplicate ac_from in user-level features
print("\n" + "=" * 80)
print("🔍 Checking for Duplicate Users in User-Level Features")
print("=" * 80)

duplicate_check_query = f"""
SELECT 
    ac_from,
    count(*) as duplicate_count
FROM public.ac_from_features_distributed
WHERE cutoff_date = toDate('{CUTOFF_DATE}')
GROUP BY ac_from
HAVING count(*) > 1
ORDER BY duplicate_count DESC
LIMIT 20
"""

try:
    duplicates = clickhouse_client.execute(duplicate_check_query)
    
    if duplicates:
        print(f"⚠️  WARNING: Found {len(duplicates)} users with duplicate records!")
        print("\nTop duplicates:")
        for ac_from, count in duplicates[:10]:
            print(f"   • ac_from: {ac_from}, count: {count}")
        
        # Get total duplicate records
        total_dup_query = f"""
        SELECT count(*) FROM (
            SELECT ac_from
            FROM public.ac_from_features_distributed
            WHERE cutoff_date = toDate('{CUTOFF_DATE}')
            GROUP BY ac_from
            HAVING count(*) > 1
        )
        """
        total_dups = clickhouse_client.execute(total_dup_query)[0][0]
        print(f"\n📊 Total users with duplicates: {total_dups:,}")
        
    else:
        print("✅ No duplicate ac_from found in user-level features!")
        print("   • Each user appears exactly once in the table")
        
except Exception as e:
    print(f"❌ Error checking for duplicates: {str(e)}")
    raise


🔍 Checking for Duplicate Users in User-Level Features
✅ No duplicate ac_from found in user-level features!
   • Each user appears exactly once in the table


## Transaction-Level Feature Engineering
Create features for each transaction using historical lookback (excluding same-day data)

In [81]:
# Drop existing transaction feature tables if they exist
drop_txn_queries = [
    "DROP TABLE IF EXISTS public.transaction_features_distributed ON CLUSTER my_cluster_2shards",
    "DROP TABLE IF EXISTS public.transaction_features_local ON CLUSTER my_cluster_2shards"
]

print("🗑️  Dropping existing transaction feature tables...")
for query in drop_txn_queries:
    try:
        clickhouse_client.execute(query)
        print(f"✅ Executed: {query.split('DROP TABLE IF EXISTS')[1].split('ON')[0].strip()}")
    except Exception as e:
        print(f"⚠️  Warning: {str(e)}")

print("\n✅ Transaction feature table cleanup complete!")

🗑️  Dropping existing transaction feature tables...
✅ Executed: public.transaction_features_distributed
✅ Executed: public.transaction_features_local

✅ Transaction feature table cleanup complete!


In [82]:
# Create transaction-level feature table
create_txn_local_table = """
CREATE TABLE IF NOT EXISTS public.transaction_features_local
ON CLUSTER my_cluster_2shards
(
    -- Identifiers
    trans_id String,
    ac_from String,
    ac_to String,
    data_date Date,
    trans_initiate_time DateTime,
    cutoff_date Date DEFAULT '2025-05-31',
    
    -- Original transaction attributes
    trx_channel String,
    trx_type String,
    start_balance Float64,
    end_balance Float64,
    trx_amt Float64,
    
    -- Time-based features
    hour_of_day UInt8,
    day_of_week UInt8,
    is_weekend UInt8,
    is_night UInt8,
    is_business_hours UInt8,
    is_unusual_hour UInt8,
    
    -- Risk indicators (current transaction)
    night_weekend_combo UInt8,
    
    -- Balance features
    start_balance_log Float64,
    balance_change Float64,
    balance_change_pct Float64,
    
    -- 3-day historical features (excluding current day)
    txns_3d UInt32 DEFAULT 0,
    total_amount_3d Float64 DEFAULT 0,
    avg_amount_3d Float64 DEFAULT 0,
    max_amount_3d Float64 DEFAULT 0,
    min_amount_3d Float64 DEFAULT 0,
    unique_recipients_3d UInt32 DEFAULT 0,
    unique_channels_3d UInt32 DEFAULT 0,
    unique_types_3d UInt32 DEFAULT 0,
    is_high_activity_3d UInt8 DEFAULT 0,
    multi_channel_recent UInt8 DEFAULT 0,
    amount_deviation_from_avg Float64 DEFAULT 0,
    night_txns_3d UInt32 DEFAULT 0,
    weekend_txns_3d UInt32 DEFAULT 0,
    
    -- Channel one-hot encoding (top channels)
    channel_new_jc_app UInt8,
    channel_ussd UInt8,
    channel_ussd_api UInt8,
    channel_payment_gateway UInt8,
    channel_mobile_app UInt8,
    
    -- Type one-hot encoding (top types)
    type_transfer_c2c UInt8,
    type_transfer_c2b UInt8,
    type_bill_payment UInt8,
    type_mobile_load UInt8,
    
    -- Metadata
    processing_timestamp DateTime DEFAULT now(),
    created_at Date DEFAULT today()
)
ENGINE = MergeTree()
PARTITION BY toYYYYMM(data_date)
ORDER BY (ac_from, trans_initiate_time, trans_id)
"""

print("📊 Creating transaction feature table: public.transaction_features_local...")
clickhouse_client.execute(create_txn_local_table)
print("✅ Transaction feature table created successfully!")
print("   • Table: public.transaction_features_local")
print("   • Engine: MergeTree")
print("   • Order By: (ac_from, trans_initiate_time, trans_id)")
print("   • Partition By: toYYYYMM(data_date)")
print("   • Features: 39 feature columns (6 current + 13 historical 3d + 20 encodings)")
print("   • Includes: cutoff_date column for filtering")

📊 Creating transaction feature table: public.transaction_features_local...
✅ Transaction feature table created successfully!
   • Table: public.transaction_features_local
   • Engine: MergeTree
   • Order By: (ac_from, trans_initiate_time, trans_id)
   • Partition By: toYYYYMM(data_date)
   • Features: 39 feature columns (6 current + 13 historical 3d + 20 encodings)
   • Includes: cutoff_date column for filtering


In [83]:
# Create distributed transaction feature table
create_txn_distributed_table = """
CREATE TABLE IF NOT EXISTS public.transaction_features_distributed AS public.transaction_features_local
ENGINE = Distributed(my_cluster_2shards, public, transaction_features_local, cityHash64(trans_id))
"""

print("\n🌐 Creating distributed transaction feature table...")
clickhouse_client.execute(create_txn_distributed_table)
print("✅ Distributed transaction feature table created successfully!")
print("   • Table: public.transaction_features_distributed")
print("   • Engine: Distributed")
print("   • Cluster: my_cluster_2shards")
print("   • Sharding Key: cityHash64(trans_id)")
print("   • Local Table: public.transaction_features_local")

print("\n" + "="*80)
print("🎉 Transaction feature table creation complete!")
print("="*80)


🌐 Creating distributed transaction feature table...
✅ Distributed transaction feature table created successfully!
   • Table: public.transaction_features_distributed
   • Engine: Distributed
   • Cluster: my_cluster_2shards
   • Sharding Key: cityHash64(trans_id)
   • Local Table: public.transaction_features_local

🎉 Transaction feature table creation complete!


In [93]:
# Configuration for transaction-level features
# Use global CUTOFF_DATE - no need to redefine
CUTOFF_DATE = '2025-06-15'  # Use the same cutoff date as user features
TXN_DATE = CUTOFF_DATE  # Use the same cutoff date as user features

print("📅 Transaction-Level Feature Engineering Configuration:")
print(f"   • Transaction Date: {TXN_DATE} (from Global Cutoff Date)")
print(f"   • Processing: 1 DAY of transactions ({TXN_DATE})")
print(f"   • Historical Lookback: 3 days (excluding current day)")
print(f"   • Feature Types: Transaction attributes, time-based, balance, 3d historical, one-hot encodings")
print(f"\n💡 Note: cutoff_date column will be added to align with user features for easy filtering")


📅 Transaction-Level Feature Engineering Configuration:
   • Transaction Date: 2025-06-15 (from Global Cutoff Date)
   • Processing: 1 DAY of transactions (2025-06-15)
   • Historical Lookback: 3 days (excluding current day)
   • Feature Types: Transaction attributes, time-based, balance, 3d historical, one-hot encodings

💡 Note: cutoff_date column will be added to align with user features for easy filtering


In [94]:
# Transaction-level feature engineering query with 3-day historical features
transaction_feature_query = f"""
INSERT INTO public.transaction_features_distributed
SELECT 
    -- Identifiers
    curr.trans_id,
    curr.ac_from,
    curr.ac_to,
    curr.data_date,
    curr.trans_initiate_time,
    toDate('{CUTOFF_DATE}') as cutoff_date,
    
    -- Original transaction attributes
    curr.trx_channel,
    curr.trx_type,
    curr.start_balance,
    curr.end_balance,
    curr.trx_amt,
    
    -- Time-based features
    toHour(curr.trans_initiate_time) as hour_of_day,
    toDayOfWeek(curr.trans_initiate_time) as day_of_week,
    if(toDayOfWeek(curr.trans_initiate_time) IN (6, 7), 1, 0) as is_weekend,
    if(toHour(curr.trans_initiate_time) >= 22 OR toHour(curr.trans_initiate_time) <= 6, 1, 0) as is_night,
    if(toHour(curr.trans_initiate_time) >= 9 AND toHour(curr.trans_initiate_time) <= 17, 1, 0) as is_business_hours,
    if(toHour(curr.trans_initiate_time) < 6 OR toHour(curr.trans_initiate_time) > 23, 1, 0) as is_unusual_hour,
    
    -- Risk indicators (current transaction)
    if((toHour(curr.trans_initiate_time) >= 22 OR toHour(curr.trans_initiate_time) <= 6) 
       AND toDayOfWeek(curr.trans_initiate_time) IN (6, 7), 1, 0) as night_weekend_combo,
    
    -- Balance features
    log(greatest(curr.start_balance, 1)) as start_balance_log,
    curr.end_balance - curr.start_balance as balance_change,
    if(curr.start_balance > 0, (curr.end_balance - curr.start_balance) / curr.start_balance, 0) as balance_change_pct,
    
    -- 3-day historical features (excluding current transaction day)
    coalesce(hist.txns_3d, 0) as txns_3d,
    coalesce(hist.total_amount_3d, 0) as total_amount_3d,
    coalesce(hist.avg_amount_3d, 0) as avg_amount_3d,
    coalesce(hist.max_amount_3d, 0) as max_amount_3d,
    coalesce(hist.min_amount_3d, 0) as min_amount_3d,
    coalesce(hist.unique_recipients_3d, 0) as unique_recipients_3d,
    coalesce(hist.unique_channels_3d, 0) as unique_channels_3d,
    coalesce(hist.unique_types_3d, 0) as unique_types_3d,
    if(coalesce(hist.txns_3d, 0) > 10, 1, 0) as is_high_activity_3d,
    if(coalesce(hist.unique_channels_3d, 0) > 1, 1, 0) as multi_channel_recent,
    if(coalesce(hist.avg_amount_3d, 0) > 0, 
       (curr.trx_amt - hist.avg_amount_3d) / hist.avg_amount_3d, 0) as amount_deviation_from_avg,
    coalesce(hist.night_txns_3d, 0) as night_txns_3d,
    coalesce(hist.weekend_txns_3d, 0) as weekend_txns_3d,
    
    -- Channel one-hot encoding
    if(curr.trx_channel = 'NEW_JC_APP', 1, 0) as channel_new_jc_app,
    if(curr.trx_channel = 'USSD', 1, 0) as channel_ussd,
    if(curr.trx_channel = 'USSD_API', 1, 0) as channel_ussd_api,
    if(curr.trx_channel = 'Payment Gateway', 1, 0) as channel_payment_gateway,
    if(curr.trx_channel = 'Mobile App', 1, 0) as channel_mobile_app,
    
    -- Type one-hot encoding
    if(curr.trx_type = 'Transfer(C2C)', 1, 0) as type_transfer_c2c,
    if(curr.trx_type = 'Transfer(C2B)', 1, 0) as type_transfer_c2b,
    if(curr.trx_type = 'Bill Payment', 1, 0) as type_bill_payment,
    if(curr.trx_type LIKE '%Load%', 1, 0) as type_mobile_load,
    
    -- Metadata
    now() as processing_timestamp,
    today() as created_at

FROM (
    SELECT *
    FROM public.stixor_iar_distributed
    WHERE data_date = toDate('{TXN_DATE}')
      AND ac_from != ''
) AS curr

GLOBAL LEFT JOIN (
    SELECT 
        ac_from,
        count() as txns_3d,
        sum(trx_amt) as total_amount_3d,
        avg(trx_amt) as avg_amount_3d,
        max(trx_amt) as max_amount_3d,
        min(trx_amt) as min_amount_3d,
        uniq(ac_to) as unique_recipients_3d,
        uniq(trx_channel) as unique_channels_3d,
        uniq(trx_type) as unique_types_3d,
        sumIf(1, toHour(trans_initiate_time) >= 22 OR toHour(trans_initiate_time) <= 6) as night_txns_3d,
        sumIf(1, toDayOfWeek(trans_initiate_time) IN (6, 7)) as weekend_txns_3d
    FROM public.stixor_iar_distributed
    WHERE data_date >= toDate('{TXN_DATE}') - INTERVAL 3 DAY
      AND data_date < toDate('{TXN_DATE}')
      AND ac_from != ''
    GROUP BY ac_from
) AS hist ON curr.ac_from = hist.ac_from
"""

print("\n📝 Transaction-level feature engineering query prepared")
print(f"   • Query length: {len(transaction_feature_query):,} characters")
print(f"   • Target table: public.transaction_features_distributed")
print(f"   • Date: {TXN_DATE}")
print(f"   • Cutoff Date: {CUTOFF_DATE}")
print(f"   • Features: 6 current + 13 historical (3d lookback) + 20 encodings = 39 features")
print(f"   • Historical window: Excludes current day (days -3 to -1)")
print(f"   • Includes cutoff_date column for filtering")


📝 Transaction-level feature engineering query prepared
   • Query length: 4,043 characters
   • Target table: public.transaction_features_distributed
   • Date: 2025-06-15
   • Cutoff Date: 2025-06-15
   • Features: 6 current + 13 historical (3d lookback) + 20 encodings = 39 features
   • Historical window: Excludes current day (days -3 to -1)
   • Includes cutoff_date column for filtering


In [95]:
# Execute transaction-level feature engineering query
print("\n" + "="*80)
print("🚀 Executing Transaction-Level Feature Engineering Query...")
print("="*80)
print("\n⏳ This may take a few minutes depending on data volume...")
print("   Please wait...")

import time
start_time = time.time()

try:
    # Execute the INSERT query
    clickhouse_client.execute(transaction_feature_query)
    
    elapsed_time = time.time() - start_time
    
    print(f"\n✅ Transaction-level feature engineering completed successfully!")
    print(f"⏱️  Execution time: {elapsed_time:.2f} seconds ({elapsed_time/60:.2f} minutes)")
    
    # Get count of inserted rows
    count_query = "SELECT count(*) FROM public.transaction_features_distributed"
    total_txns = clickhouse_client.execute(count_query)[0][0]
    
    print(f"\n📊 Results:")
    print(f"   • Total transactions processed: {total_txns:,}")
    print(f"   • Features per transaction: 39 (6 current + 13 historical 3d + 20 encodings)")
    print(f"   • Total feature values: {total_txns * 39:,}")
    print(f"\n💡 Note: Each transaction enriched with 3-day historical patterns (excluding same day)")
    
except Exception as e:
    elapsed_time = time.time() - start_time
    print(f"\n❌ Transaction feature engineering failed after {elapsed_time:.2f} seconds")
    print(f"   Error: {str(e)}")
    raise


🚀 Executing Transaction-Level Feature Engineering Query...

⏳ This may take a few minutes depending on data volume...
   Please wait...

✅ Transaction-level feature engineering completed successfully!
⏱️  Execution time: 10.52 seconds (0.18 minutes)

📊 Results:
   • Total transactions processed: 39,127,527
   • Features per transaction: 39 (6 current + 13 historical 3d + 20 encodings)
   • Total feature values: 1,525,973,553

💡 Note: Each transaction enriched with 3-day historical patterns (excluding same day)

✅ Transaction-level feature engineering completed successfully!
⏱️  Execution time: 10.52 seconds (0.18 minutes)

📊 Results:
   • Total transactions processed: 39,127,527
   • Features per transaction: 39 (6 current + 13 historical 3d + 20 encodings)
   • Total feature values: 1,525,973,553

💡 Note: Each transaction enriched with 3-day historical patterns (excluding same day)


In [96]:
# Check for duplicate trans_id in transaction-level features
print("\n" + "=" * 80)
print("🔍 Checking for Duplicate Transactions in Transaction-Level Features")
print("=" * 80)

duplicate_txn_check_query = """
SELECT 
    trans_id,
    count(*) as duplicate_count
FROM public.transaction_features_distributed
GROUP BY trans_id
HAVING count(*) > 1
ORDER BY duplicate_count DESC
LIMIT 20
"""

try:
    duplicates_txn = clickhouse_client.execute(duplicate_txn_check_query)
    
    if duplicates_txn:
        print(f"⚠️  WARNING: Found {len(duplicates_txn)} transactions with duplicate records!")
        print("\nTop duplicates:")
        for trans_id, count in duplicates_txn[:10]:
            print(f"   • trans_id: {trans_id}, count: {count}")
        
        # Get total duplicate records
        total_dup_txn_query = """
        SELECT count(*) FROM (
            SELECT trans_id
            FROM public.transaction_features_distributed
            GROUP BY trans_id
            HAVING count(*) > 1
        )
        """
        total_dups_txn = clickhouse_client.execute(total_dup_txn_query)[0][0]
        print(f"\n📊 Total transactions with duplicates: {total_dups_txn:,}")
        
        # Also show how many extra rows this creates
        extra_rows_query = """
        SELECT sum(cnt - 1) FROM (
            SELECT count(*) as cnt
            FROM public.transaction_features_distributed
            GROUP BY trans_id
            HAVING count(*) > 1
        )
        """
        extra_rows = clickhouse_client.execute(extra_rows_query)[0][0]
        if extra_rows:
            print(f"📊 Extra rows due to duplicates: {extra_rows:,}")
        
    else:
        print("✅ No duplicate trans_id found in transaction-level features!")
        print("   • Each transaction appears exactly once in the table")
        
except Exception as e:
    print(f"❌ Error checking for duplicates: {str(e)}")
    raise


🔍 Checking for Duplicate Transactions in Transaction-Level Features
✅ No duplicate trans_id found in transaction-level features!
   • Each transaction appears exactly once in the table
✅ No duplicate trans_id found in transaction-level features!
   • Each transaction appears exactly once in the table


## Combined Features
Include ALL 38 user-level features (9 x 3d + 29 x 7d) instead of just 27

In [97]:
# Drop existing tables to recreate with ALL user features
drop_corrected_combined = [
    "DROP TABLE IF EXISTS public.combined_features_distributed ON CLUSTER my_cluster_2shards",
    "DROP TABLE IF EXISTS public.combined_features_local ON CLUSTER my_cluster_2shards"
]

print("🗑️  Dropping combined feature tables to add ALL user-level features (3d + 7d)...")
for query in drop_corrected_combined:
    try:
        clickhouse_client.execute(query)
        print(f"✅ Executed: {query.split('DROP TABLE IF EXISTS')[1].split('ON')[0].strip()}")
    except Exception as e:
        print(f"⚠️  Warning: {str(e)}")

print("\n✅ Cleanup complete! Ready to create corrected table with ALL user features.")

🗑️  Dropping combined feature tables to add ALL user-level features (3d + 7d)...
✅ Executed: public.combined_features_distributed
✅ Executed: public.combined_features_local

✅ Cleanup complete! Ready to create corrected table with ALL user features.


In [98]:
# Create corrected combined features table with ALL user features (3d + 7d)
create_corrected_combined_table = """
CREATE TABLE IF NOT EXISTS public.combined_features_local
ON CLUSTER my_cluster_2shards
(
    -- Transaction identifiers
    trans_id String,
    ac_from String,
    ac_to String,
    data_date Date,
    trans_initiate_time DateTime,
    cutoff_date Date DEFAULT '2025-07-01',
    
    -- FRAUD LABEL (Target Variable)
    fraud_flag UInt8 DEFAULT 0,
    
    -- Original transaction attributes
    trx_channel String,
    trx_type String,
    start_balance Float64,
    end_balance Float64,
    trx_amt Float64,
    
    -- MBAR Account Information (ALL 24 columns from stixor_mbar_v)
    mbar_a_c_reference String DEFAULT '',
    mbar_region String DEFAULT '',
    mbar_city String DEFAULT '',
    mbar_registered_channel String DEFAULT '',
    mbar_registered_date_time DateTime DEFAULT toDateTime('1970-01-01 00:00:00'),
    mbar_a_c_status String DEFAULT '',
    mbar_a_c_level String DEFAULT '',
    mbar_agent_group String DEFAULT '',
    mbar_limit_group String DEFAULT '',
    mbar_charge_profile String DEFAULT '',
    mbar_credit_dl_ml_yl Decimal(18, 2) DEFAULT 0,
    mbar_debit_dl_ml_yl Decimal(18, 2) DEFAULT 0,
    mbar_year_of_birth Int32 DEFAULT 0,
    mbar_last_modified_date_time DateTime DEFAULT toDateTime('1970-01-01 00:00:00'),
    mbar_dormant_date Date DEFAULT toDate('1970-01-01'),
    mbar_re_active_date Date DEFAULT toDate('1970-01-01'),
    mbar_place_of_birth String DEFAULT '',
    mbar_account_type_name String DEFAULT '',
    mbar_mpin_status String DEFAULT '',
    mbar_filer String DEFAULT '',
    mbar_prov String DEFAULT '',
    mbar_year_mdob String DEFAULT '',
    mbar_gmsisdn String DEFAULT '',
    mbar_trust_level String DEFAULT '',
    
    -- Transaction-level time-based features
    hour_of_day UInt8,
    day_of_week UInt8,
    is_weekend UInt8,
    is_night UInt8,
    is_business_hours UInt8,
    is_unusual_hour UInt8,
    
    -- Transaction-level risk indicators
    night_weekend_combo UInt8,
    
    -- Transaction-level balance features
    start_balance_log Float64,
    balance_change Float64,
    balance_change_pct Float64,
    
    -- Transaction-level 3-day historical features
    txn_txns_3d UInt32,
    txn_total_amount_3d Float64,
    txn_avg_amount_3d Float64,
    txn_max_amount_3d Float64,
    txn_min_amount_3d Float64,
    txn_unique_recipients_3d UInt32,
    txn_unique_channels_3d UInt32,
    txn_unique_types_3d UInt32,
    txn_is_high_activity_3d UInt8,
    txn_multi_channel_recent UInt8,
    txn_amount_deviation_from_avg Float64,
    txn_night_txns_3d UInt32,
    txn_weekend_txns_3d UInt32,
    
    -- Transaction-level channel one-hot encoding
    channel_new_jc_app UInt8,
    channel_ussd UInt8,
    channel_ussd_api UInt8,
    channel_payment_gateway UInt8,
    channel_mobile_app UInt8,
    
    -- Transaction-level type one-hot encoding
    type_transfer_c2c UInt8,
    type_transfer_c2b UInt8,
    type_bill_payment UInt8,
    type_mobile_load UInt8,
    
    -- User-level 3-day aggregate features (9 features)
    user_total_txns_3d UInt32,
    user_total_amount_3d Float64,
    user_avg_amount_3d Float64,
    user_median_amount_3d Float64,
    user_max_amount_3d Float64,
    user_min_amount_3d Float64,
    user_unique_recipients_3d UInt32,
    user_unique_channels_3d UInt32,
    user_unique_types_3d UInt32,
    
    -- User-level 7-day aggregate features (9 features)
    user_total_txns_7d UInt32,
    user_total_amount_7d Float64,
    user_avg_amount_7d Float64,
    user_median_amount_7d Float64,
    user_max_amount_7d Float64,
    user_min_amount_7d Float64,
    user_unique_recipients_7d UInt32,
    user_unique_channels_7d UInt32,
    user_unique_types_7d UInt32,
    
    -- User-level channel features (7-day)
    user_most_used_channel_7d String,
    user_last_used_channel String,
    user_channel_diversity_score_7d Float64,
    
    -- User-level type features (7-day)
    user_most_used_type_7d String,
    user_last_used_type String,
    user_type_diversity_score_7d Float64,
    
    -- User-level time-based features (7-day)
    user_night_txns_7d UInt32,
    user_weekend_txns_7d UInt32,
    user_peak_hour_txns_7d UInt32,
    user_off_peak_hour_txns_7d UInt32,
    
    -- User-level balance features (7-day)
    user_avg_start_balance_7d Float64,
    user_avg_end_balance_7d Float64,
    user_min_balance_7d Float64,
    user_max_balance_7d Float64,
    user_balance_volatility_7d Float64,
    
    -- User-level recipient features (7-day)
    user_top_recipient_7d String,
    user_avg_amount_per_recipient_7d Float64,
    user_max_amount_to_single_recipient_7d Float64,
    user_recipient_concentration_ratio_7d Float64,
    
    -- User-level behavioral features (7-day)
    user_avg_time_between_txns_7d Float64,
    user_txn_frequency_score_7d Float64,
    user_first_txn_time DateTime,
    user_last_txn_time DateTime,
    user_days_since_last_txn UInt32,
    
    -- Metadata
    processing_timestamp DateTime DEFAULT now(),
    created_at Date DEFAULT today()
)
ENGINE = MergeTree()
PARTITION BY toYYYYMM(data_date)
ORDER BY (ac_from, trans_initiate_time, trans_id)
"""

print("📊 Creating corrected combined features table with ALL user features...")
clickhouse_client.execute(create_corrected_combined_table)
print("✅ Combined features table created successfully!")
print("   • Table: public.combined_features_local")
print("   • Engine: MergeTree")
print("   • Order By: (ac_from, trans_initiate_time, trans_id)")
print("   • Partition By: toYYYYMM(data_date)")
print("   • 🎯 FRAUD FLAG: fraud_flag column (1 = fraud, 0 = legitimate)")
print("   • Transaction features: 39 columns")
print("   • User-level 3d features: 9 columns")
print("   • User-level 7d features: 29 columns")
print("   • MBAR features: 24 columns")
print("   • Total: 39 + 9 + 29 + 24 + 1 = 102 columns (101 features + 1 fraud label)")

📊 Creating corrected combined features table with ALL user features...
✅ Combined features table created successfully!
   • Table: public.combined_features_local
   • Engine: MergeTree
   • Order By: (ac_from, trans_initiate_time, trans_id)
   • Partition By: toYYYYMM(data_date)
   • 🎯 FRAUD FLAG: fraud_flag column (1 = fraud, 0 = legitimate)
   • Transaction features: 39 columns
   • User-level 3d features: 9 columns
   • User-level 7d features: 29 columns
   • MBAR features: 24 columns
   • Total: 39 + 9 + 29 + 24 + 1 = 102 columns (101 features + 1 fraud label)


In [99]:
# Create distributed table
create_corrected_distributed = """
CREATE TABLE IF NOT EXISTS public.combined_features_distributed AS public.combined_features_local
ENGINE = Distributed(my_cluster_2shards, public, combined_features_local, cityHash64(trans_id))
"""

print("\n🌐 Creating distributed combined features table...")
clickhouse_client.execute(create_corrected_distributed)
print("✅ Distributed combined features table created successfully!")
print("   • Table: public.combined_features_distributed")
print("   • Engine: Distributed")
print("   • Cluster: my_cluster_2shards")
print("   • Sharding Key: cityHash64(trans_id)")

print("\n" + "="*80)
print("🎉 Corrected table structure ready with ALL user features!")
print("="*80)


🌐 Creating distributed combined features table...
✅ Distributed combined features table created successfully!
   • Table: public.combined_features_distributed
   • Engine: Distributed
   • Cluster: my_cluster_2shards
   • Sharding Key: cityHash64(trans_id)

🎉 Corrected table structure ready with ALL user features!


In [117]:
CUTOFF_DATE='2025-06-15'

In [118]:
print(CUTOFF_DATE)

2025-06-15


In [119]:
# CORRECTED: Final combined query with ALL user features (3d + 7d), MBAR, and fraud labels
corrected_final_query = f"""
INSERT INTO public.combined_features_distributed
SELECT 
    -- Transaction identifiers
    t.trans_id,
    t.ac_from,
    t.ac_to,
    t.data_date,
    t.trans_initiate_time,
    toDate('{CUTOFF_DATE}') as cutoff_date,
    
    -- FRAUD LABEL: 1 if trans_id exists in fraud table, 0 otherwise
    -- FIXED: Check for empty string ('') instead of NULL because ClickHouse LEFT JOIN returns '' for non-matches
    if(f.trans_id != '', 1, 0) as fraud_flag,
    
    -- Original transaction attributes
    t.trx_channel,
    t.trx_type,
    t.start_balance,
    t.end_balance,
    t.trx_amt,
    
    -- MBAR Account Information (ALL 24 columns)
    coalesce(m.a_c_reference, '') as mbar_a_c_reference,
    coalesce(m.region, '') as mbar_region,
    coalesce(m.city, '') as mbar_city,
    coalesce(m.registered_channel, '') as mbar_registered_channel,
    coalesce(m.registered_date_time, toDateTime('1970-01-01 00:00:00')) as mbar_registered_date_time,
    coalesce(m.a_c_status, '') as mbar_a_c_status,
    coalesce(m.a_c_level, '') as mbar_a_c_level,
    coalesce(m.agent_group, '') as mbar_agent_group,
    coalesce(m.limit_group, '') as mbar_limit_group,
    coalesce(m.charge_profile, '') as mbar_charge_profile,
    coalesce(m.credit_dl_ml_yl, 0) as mbar_credit_dl_ml_yl,
    coalesce(m.debit_dl_ml_yl, 0) as mbar_debit_dl_ml_yl,
    coalesce(m.year_of_birth, 0) as mbar_year_of_birth,
    coalesce(m.last_modified_date_time, toDateTime('1970-01-01 00:00:00')) as mbar_last_modified_date_time,
    coalesce(m.dormant_date, toDate('1970-01-01')) as mbar_dormant_date,
    coalesce(m.re_active_date, toDate('1970-01-01')) as mbar_re_active_date,
    coalesce(m.place_of_birth, '') as mbar_place_of_birth,
    coalesce(m.account_type_name, '') as mbar_account_type_name,
    coalesce(m.mpin_status, '') as mbar_mpin_status,
    coalesce(m.filer, '') as mbar_filer,
    coalesce(m.prov, '') as mbar_prov,
    coalesce(m.year_mdob, '') as mbar_year_mdob,
    coalesce(m.gmsisdn, '') as mbar_gmsisdn,
    coalesce(m.trust_level, '') as mbar_trust_level,
    
    -- Transaction-level time-based features
    t.hour_of_day,
    t.day_of_week,
    t.is_weekend,
    t.is_night,
    t.is_business_hours,
    t.is_unusual_hour,
    
    -- Transaction-level risk indicators
    t.night_weekend_combo,
    
    -- Transaction-level balance features
    t.start_balance_log,
    t.balance_change,
    t.balance_change_pct,
    
    -- Transaction-level 3-day historical features
    t.txns_3d as txn_txns_3d,
    t.total_amount_3d as txn_total_amount_3d,
    t.avg_amount_3d as txn_avg_amount_3d,
    t.max_amount_3d as txn_max_amount_3d,
    t.min_amount_3d as txn_min_amount_3d,
    t.unique_recipients_3d as txn_unique_recipients_3d,
    t.unique_channels_3d as txn_unique_channels_3d,
    t.unique_types_3d as txn_unique_types_3d,
    t.is_high_activity_3d as txn_is_high_activity_3d,
    t.multi_channel_recent as txn_multi_channel_recent,
    t.amount_deviation_from_avg as txn_amount_deviation_from_avg,
    t.night_txns_3d as txn_night_txns_3d,
    t.weekend_txns_3d as txn_weekend_txns_3d,
    
    -- Transaction-level channel one-hot encoding
    t.channel_new_jc_app,
    t.channel_ussd,
    t.channel_ussd_api,
    t.channel_payment_gateway,
    t.channel_mobile_app,
    
    -- Transaction-level type one-hot encoding
    t.type_transfer_c2c,
    t.type_transfer_c2b,
    t.type_bill_payment,
    t.type_mobile_load,
    
    -- User-level 3-day aggregate features (9 features)
    coalesce(u.total_txns_3d, 0) as user_total_txns_3d,
    coalesce(u.total_amount_3d, 0) as user_total_amount_3d,
    coalesce(u.avg_amount_3d, 0) as user_avg_amount_3d,
    coalesce(u.median_amount_3d, 0) as user_median_amount_3d,
    coalesce(u.max_amount_3d, 0) as user_max_amount_3d,
    coalesce(u.min_amount_3d, 0) as user_min_amount_3d,
    coalesce(u.unique_recipients_3d, 0) as user_unique_recipients_3d,
    coalesce(u.unique_channels_3d, 0) as user_unique_channels_3d,
    coalesce(u.unique_types_3d, 0) as user_unique_types_3d,
    
    -- User-level 7-day aggregate features (9 features)
    coalesce(u.total_txns_7d, 0) as user_total_txns_7d,
    coalesce(u.total_amount_7d, 0) as user_total_amount_7d,
    coalesce(u.avg_amount_7d, 0) as user_avg_amount_7d,
    coalesce(u.median_amount_7d, 0) as user_median_amount_7d,
    coalesce(u.max_amount_7d, 0) as user_max_amount_7d,
    coalesce(u.min_amount_7d, 0) as user_min_amount_7d,
    coalesce(u.unique_recipients_7d, 0) as user_unique_recipients_7d,
    coalesce(u.unique_channels_7d, 0) as user_unique_channels_7d,
    coalesce(u.unique_types_7d, 0) as user_unique_types_7d,
    
    -- User-level channel features (7-day)
    coalesce(u.most_used_channel_7d, '') as user_most_used_channel_7d,
    coalesce(u.last_used_channel, '') as user_last_used_channel,
    coalesce(u.channel_diversity_score_7d, 0) as user_channel_diversity_score_7d,
    
    -- User-level type features (7-day)
    coalesce(u.most_used_type_7d, '') as user_most_used_type_7d,
    coalesce(u.last_used_type, '') as user_last_used_type,
    coalesce(u.type_diversity_score_7d, 0) as user_type_diversity_score_7d,
    
    -- User-level time-based features (7-day)
    coalesce(u.night_txns_7d, 0) as user_night_txns_7d,
    coalesce(u.weekend_txns_7d, 0) as user_weekend_txns_7d,
    coalesce(u.peak_hour_txns_7d, 0) as user_peak_hour_txns_7d,
    coalesce(u.off_peak_hour_txns_7d, 0) as user_off_peak_hour_txns_7d,
    
    -- User-level balance features (7-day)
    coalesce(u.avg_start_balance_7d, 0) as user_avg_start_balance_7d,
    coalesce(u.avg_end_balance_7d, 0) as user_avg_end_balance_7d,
    coalesce(u.min_balance_7d, 0) as user_min_balance_7d,
    coalesce(u.max_balance_7d, 0) as user_max_balance_7d,
    coalesce(u.balance_volatility_7d, 0) as user_balance_volatility_7d,
    
    -- User-level recipient features (7-day)
    coalesce(u.top_recipient_7d, '') as user_top_recipient_7d,
    coalesce(u.avg_amount_per_recipient_7d, 0) as user_avg_amount_per_recipient_7d,
    coalesce(u.max_amount_to_single_recipient_7d, 0) as user_max_amount_to_single_recipient_7d,
    coalesce(u.recipient_concentration_ratio_7d, 0) as user_recipient_concentration_ratio_7d,
    
    -- User-level behavioral features (7-day)
    coalesce(u.avg_time_between_txns_7d, 0) as user_avg_time_between_txns_7d,
    coalesce(u.txn_frequency_score_7d, 0) as user_txn_frequency_score_7d,
    coalesce(u.first_txn_time, toDateTime('1970-01-01 00:00:00')) as user_first_txn_time,
    coalesce(u.last_txn_time, toDateTime('1970-01-01 00:00:00')) as user_last_txn_time,
    coalesce(u.days_since_last_txn, 0) as user_days_since_last_txn,
    
    -- Metadata
    now() as processing_timestamp,
    today() as created_at

FROM (SELECT * FROM public.transaction_features_distributed 
WHERE cutoff_date = toDate('{CUTOFF_DATE}') ) AS t

-- Join with user-level features (ALL 38 features: 9 x 3d + 29 x 7d)
GLOBAL LEFT JOIN public.ac_from_features_distributed AS u
    ON t.ac_from = u.ac_from
    AND u.cutoff_date = toDate('{CUTOFF_DATE}')

-- Join with MBAR data (24 columns)
GLOBAL LEFT JOIN (
    SELECT DISTINCT
        a_c_reference,
        region,
        city,
        registered_channel,
        registered_date_time,
        a_c_status,
        a_c_level,
        agent_group,
        limit_group,
        charge_profile,
        credit_dl_ml_yl,
        debit_dl_ml_yl,
        year_of_birth,
        last_modified_date_time,
        dormant_date,
        re_active_date,
        place_of_birth,
        account_type_name,
        mpin_status,
        filer,
        prov,
        year_mdob,
        gmsisdn,
        trust_level
    FROM public.stixor_mbar_v
) AS m
    ON t.ac_from = m.a_c_reference

-- Join with fraud labels
GLOBAL LEFT JOIN (
    SELECT DISTINCT trans_id
    FROM public.fraud
) AS f
    ON t.trans_id = f.trans_id
"""

print("\n📝 CORRECTED combined query with ALL user features prepared")
print(f"   • Query length: {len(corrected_final_query):,} characters")
print(f"   • Target table: public.combined_features_distributed")
print(f"   • Cutoff date filter: {CUTOFF_DATE}")
print(f"\n   🔗 Join Operations:")
print(f"     1. transaction_features ⟶ ac_from_features (ALL 38 user features) [filtered by cutoff_date]")
print(f"     2. transaction_features ⟶ stixor_mbar_v (24 MBAR columns)")
print(f"     3. transaction_features ⟶ fraud (fraud labels)")
print(f"\n   📊 Complete Feature Summary:")
print(f"     • Fraud Flag: 1 column (target)")
print(f"     • Transaction features: 39 columns")
print(f"     • User 3d features: 9 columns")
print(f"     • User 7d features: 29 columns (NOW INCLUDED!)")
print(f"     • MBAR features: 24 columns")
print(f"     • Total: 1 + 39 + 9 + 29 + 24 = 102 columns")
print(f"\n   ✅ FIXED: Now includes ALL user-level features (both 3d and 7d)")
print(f"   ✅ FILTERED: Both transaction and user features filtered by cutoff_date = {CUTOFF_DATE}")


📝 CORRECTED combined query with ALL user features prepared
   • Query length: 7,850 characters
   • Target table: public.combined_features_distributed
   • Cutoff date filter: 2025-06-15

   🔗 Join Operations:
     1. transaction_features ⟶ ac_from_features (ALL 38 user features) [filtered by cutoff_date]
     2. transaction_features ⟶ stixor_mbar_v (24 MBAR columns)
     3. transaction_features ⟶ fraud (fraud labels)

   📊 Complete Feature Summary:
     • Fraud Flag: 1 column (target)
     • Transaction features: 39 columns
     • User 3d features: 9 columns
     • User 7d features: 29 columns (NOW INCLUDED!)
     • MBAR features: 24 columns
     • Total: 1 + 39 + 9 + 29 + 24 = 102 columns

   ✅ FIXED: Now includes ALL user-level features (both 3d and 7d)
   ✅ FILTERED: Both transaction and user features filtered by cutoff_date = 2025-06-15


In [120]:
# print(corrected_final_query)

In [121]:
# Execute CORRECTED final combined feature engineering
print("\n" + "="*80)
print("🚀 Executing CORRECTED Combined Feature Engineering")
print("    WITH ALL USER FEATURES (3d + 7d) + MBAR + FRAUD LABELS")
print("="*80)
print("\n⏳ This may take several minutes to join all tables...")
print("   • Joining transaction features (39 cols)")
print("   • Joining user-level 3d features (9 cols)")
print("   • Joining user-level 7d features (29 cols) ✅ NOW INCLUDED")
print("   • Joining MBAR account data (24 cols)")
print("   • Joining fraud labels (1 col)")
print("\n   Total: 102 columns")
print("\n   Please wait...")

import time
start_time = time.time()

try:
    # Execute the INSERT\ query
    clickhouse_client.execute(corrected_final_query)
    
    elapsed_time = time.time() - start_time
    
    print(f"\n✅ CORRECTED combined feature engineering completed successfully!")
    print(f"⏱️  Execution time: {elapsed_time:.2f} seconds ({elapsed_time/60:.2f} minutes)")
    
    # Get count of inserted rows
    count_query = f"SELECT count(*) FROM public.combined_features_distributed where cutoff_date='{CUTOFF_DATE}' "
    total_records = clickhouse_client.execute(count_query)[0][0]
    
    # Get fraud statistics
    fraud_stats_query = """
    SELECT 
        countIf(fraud_flag = 1) as fraud_count,
        countIf(fraud_flag = 0) as legit_count,
        round(countIf(fraud_flag = 1) * 100.0 / count(*), 2) as fraud_rate
    FROM public.combined_features_distributed
    """
    fraud_stats = clickhouse_client.execute(fraud_stats_query)[0]
    
    print(f"\n📊 Results Summary:")
    print(f"   • Total records: {total_records:,}")
    print(f"   • Fraudulent transactions: {fraud_stats[0]:,} (fraud_flag = 1)")
    print(f"   • Legitimate transactions: {fraud_stats[1]:,} (fraud_flag = 0)")
    print(f"   • Fraud rate: {fraud_stats[2]:.2f}%")
    
    print(f"\n📋 Complete Feature Summary:")
    print(f"   ✅ Fraud Flag: 1 column (target variable)")
    print(f"   ✅ Transaction features: 39 columns")
    print(f"   ✅ User 3d features: 9 columns")
    print(f"   ✅ User 7d features: 29 columns (INCLUDED!)")
    print(f"   ✅ MBAR features: 24 columns")
    print(f"   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━")
    print(f"   ✅ TOTAL: 102 columns (101 features + 1 label)")
    
    print(f"\n💾 Total feature values: {total_records * 102:,}")
    
    print(f"\n🎯 Dataset Ready For:")
    print(f"   ✓ Machine learning model training (supervised)")
    print(f"   ✓ Fraud pattern analysis with 3d and 7d trends")
    print(f"   ✓ Feature importance analysis (101 features)")
    print(f"   ✓ Model evaluation with real fraud labels")
    print(f"   ✓ Time-window comparison (3d vs 7d patterns)")
    
except Exception as e:
    elapsed_time = time.time() - start_time
    print(f"\n❌ Feature engineering failed after {elapsed_time:.2f} seconds")
    print(f"   Error: {str(e)}")
    raise


🚀 Executing CORRECTED Combined Feature Engineering
    WITH ALL USER FEATURES (3d + 7d) + MBAR + FRAUD LABELS

⏳ This may take several minutes to join all tables...
   • Joining transaction features (39 cols)
   • Joining user-level 3d features (9 cols)
   • Joining user-level 7d features (29 cols) ✅ NOW INCLUDED
   • Joining MBAR account data (24 cols)
   • Joining fraud labels (1 col)

   Total: 102 columns

   Please wait...

✅ CORRECTED combined feature engineering completed successfully!
⏱️  Execution time: 75.00 seconds (1.25 minutes)

📊 Results Summary:
   • Total records: 8,542,113
   • Fraudulent transactions: 649 (fraud_flag = 1)
   • Legitimate transactions: 39,705,228 (fraud_flag = 0)
   • Fraud rate: 0.00%

📋 Complete Feature Summary:
   ✅ Fraud Flag: 1 column (target variable)
   ✅ Transaction features: 39 columns
   ✅ User 3d features: 9 columns
   ✅ User 7d features: 29 columns (INCLUDED!)
   ✅ MBAR features: 24 columns
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━
   ✅ TOTAL: 102 c

## Step 5: Class Imbalance Handling

Apply undersampling techniques to create balanced datasets for training fraud detection models.

**Strategies:**
1. **Random Undersampling** - Randomly sample non-fraud cases
2. **Stratified Sampling** - Multiple ratios (1:1, 1:5, 1:10, 1:20)
3. **All Fraud + Sample** - Keep all fraud cases and sample legitimate cases

In [122]:
# Check current class distribution
print("📊 Current Class Distribution")
print("=" * 70)

class_dist_query = """
SELECT 
    fraud_flag,
    count(*) as count,
    round(count(*) * 100.0 / sum(count(*)) OVER (), 4) as percentage
FROM public.combined_features_distributed
GROUP BY fraud_flag
ORDER BY fraud_flag
"""

distribution = clickhouse_client.execute(class_dist_query)

total_records = sum(row[1] for row in distribution)
fraud_count = next((row[1] for row in distribution if row[0] == 1), 0)
legit_count = next((row[1] for row in distribution if row[0] == 0), 0)

print(f"\n📈 Class Distribution:")
for fraud_flag, count, pct in distribution:
    label = "🚨 FRAUD" if fraud_flag == 1 else "✅ LEGITIMATE"
    print(f"   {label}: {count:,} ({pct}%)")

print(f"\n📉 Imbalance Metrics:")
print(f"   • Total records: {total_records:,}")
print(f"   • Fraud cases: {fraud_count:,}")
print(f"   • Legitimate cases: {legit_count:,}")
print(f"   • Imbalance ratio: 1:{int(legit_count/fraud_count):,}")
print(f"   • Fraud percentage: {(fraud_count/total_records)*100:.4f}%")

print("\n" + "=" * 70)

📊 Current Class Distribution

📈 Class Distribution:
   ✅ LEGITIMATE: 39,923,401 (99.9984%)
   🚨 FRAUD: 654 (0.0016%)

📉 Imbalance Metrics:
   • Total records: 39,924,055
   • Fraud cases: 654
   • Legitimate cases: 39,923,401
   • Imbalance ratio: 1:61,044
   • Fraud percentage: 0.0016%



In [123]:
# Create balanced dataset table with different sampling ratios
print("🎯 Creating Balanced Dataset Tables")
print("=" * 70)

# Define sampling strategies
sampling_strategies = [
    # {"name": "balanced_1_1", "ratio": 1, "description": "1:1 ratio (equal fraud and legitimate)"},
    # {"name": "balanced_1_5", "ratio": 5, "description": "1:5 ratio (5 legitimate per fraud)"},
    # {"name": "balanced_1_10", "ratio": 10, "description": "1:10 ratio (10 legitimate per fraud)"},
    {"name": "balanced_1_20", "ratio": 20, "description": "1:20 ratio (20 legitimate per fraud)"},
    {"name": "balanced_1_100", "ratio": 100, "description": "1:100 ratio (100 legitimate per fraud)"},
    {"name": "balanced_1_1000", "ratio": 1000, "description": "1:1000 ratio (1000 legitimate per fraud)"},
    {"name": "balanced_1_10000", "ratio": 10000, "description": "1:10000 ratio (10000 legitimate per fraud)"},
    {"name": "balanced_1_100000", "ratio": 100000, "description": "1:100000 ratio (100000 legitimate per fraud)"},
]

print(f"\n📋 Sampling Strategies:")
for i, strategy in enumerate(sampling_strategies, 1):
    print(f"   {i}. {strategy['name']}: {strategy['description']}")
    
print(f"\n✅ Will create {len(sampling_strategies)} balanced dataset tables")
print("=" * 70)

🎯 Creating Balanced Dataset Tables

📋 Sampling Strategies:
   1. balanced_1_20: 1:20 ratio (20 legitimate per fraud)
   2. balanced_1_100: 1:100 ratio (100 legitimate per fraud)
   3. balanced_1_1000: 1:1000 ratio (1000 legitimate per fraud)
   4. balanced_1_10000: 1:10000 ratio (10000 legitimate per fraud)
   5. balanced_1_100000: 1:100000 ratio (100000 legitimate per fraud)

✅ Will create 5 balanced dataset tables


In [139]:
# Create DDL for balanced dataset tables
print("📝 Creating Table Schemas for Balanced Datasets")
print("=" * 70)

for strategy in sampling_strategies:
    table_name = f"combined_features_{strategy['name']}"
    
    # Drop existing tables
    drop_queries = [
        f"DROP TABLE IF EXISTS public.{table_name}_distributed ON CLUSTER my_cluster_2shards",
        f"DROP TABLE IF EXISTS public.{table_name}_local ON CLUSTER my_cluster_2shards"
    ]
    
    for drop_query in drop_queries:
        try:
            clickhouse_client.execute(drop_query)
        except Exception as e:
            pass  # Ignore if table doesn't exist
    
    # Create local table (same schema as combined_features_local)
    create_local = f"""
    CREATE TABLE IF NOT EXISTS public.{table_name}_local ON CLUSTER my_cluster_2shards
    (
        trans_id String,
        data_date Date,
        cutoff_date Date,
        fraud_flag UInt8,
        trx_channel String,
        trx_type String,
        mbar_account_type_name String
    )
    ENGINE = MergeTree()
    ORDER BY (fraud_flag, trans_id)
    """
    
    clickhouse_client.execute(create_local)
    
    # Create distributed table
    create_distributed = f"""
    CREATE TABLE IF NOT EXISTS public.{table_name}_distributed ON CLUSTER my_cluster_2shards
    AS public.{table_name}_local
    ENGINE = Distributed(my_cluster_2shards, public, {table_name}_local, cityHash64(trans_id))
    """
    
    clickhouse_client.execute(create_distributed)
    
    print(f"✅ Created tables for: {strategy['name']}")

print("\n✅ All balanced dataset table schemas created!")
print("=" * 70)

📝 Creating Table Schemas for Balanced Datasets
✅ Created tables for: balanced_1_20
✅ Created tables for: balanced_1_100
✅ Created tables for: balanced_1_1000
✅ Created tables for: balanced_1_10000
✅ Created tables for: balanced_1_100000

✅ All balanced dataset table schemas created!


In [140]:
# Populate balanced datasets using random undersampling
print("⚖️  Populating Balanced Datasets with Random Undersampling")
print("=" * 70)

import time

for strategy in sampling_strategies:
    table_name = f"combined_features_{strategy['name']}"
    ratio = strategy['ratio']
    
    print(f"\n🔄 Processing: {strategy['name']} ({strategy['description']})")
    
    # Calculate sample size for legitimate cases
    legit_sample_size = fraud_count * ratio
    
    # Random sampling using ClickHouse's SAMPLE clause
    # Get all fraud cases + random sample of legitimate cases

    insert_query = f"""
    INSERT INTO public.{table_name}_distributed
    SELECT trans_id, data_date, cutoff_date, fraud_flag, trx_channel, trx_type, mbar_account_type_name FROM (
        -- All fraud cases
        SELECT trans_id, data_date, cutoff_date, fraud_flag, trx_channel, trx_type, mbar_account_type_name FROM public.combined_features_distributed
        WHERE fraud_flag = 1
        
        UNION ALL
        
        -- Random sample of legitimate cases
        SELECT trans_id, data_date, cutoff_date, fraud_flag, trx_channel, trx_type, mbar_account_type_name FROM public.combined_features_distributed
        WHERE fraud_flag = 0
        ORDER BY rand()
        LIMIT {legit_sample_size}
    )
    """
    
    start_time = time.time()
    clickhouse_client.execute(insert_query)
    elapsed = time.time() - start_time
    
    # Verify the result
    verify_query = f"""
    SELECT 
        fraud_flag,
        count(*) as count
    FROM public.{table_name}_distributed
    GROUP BY fraud_flag
    ORDER BY fraud_flag
    """
    
    result = clickhouse_client.execute(verify_query)
    
    legit = next((r[1] for r in result if r[0] == 0), 0)
    fraud = next((r[1] for r in result if r[0] == 1), 0)
    total = legit + fraud
    
    print(f"   ✅ Completed in {elapsed:.2f}s")
    print(f"   📊 Result: {fraud:,} fraud + {legit:,} legitimate = {total:,} total")
    print(f"   📈 Actual ratio: 1:{int(legit/fraud) if fraud > 0 else 0}")
    print(f"   💾 Reduction: {total_records:,} → {total:,} ({(total/total_records)*100:.2f}%)")

print("\n" + "=" * 70)
print("✅ All balanced datasets created successfully!")
print("=" * 70)

⚖️  Populating Balanced Datasets with Random Undersampling

🔄 Processing: balanced_1_20 (1:20 ratio (20 legitimate per fraud))
   ✅ Completed in 0.23s
   📊 Result: 654 fraud + 13,080 legitimate = 13,734 total
   📈 Actual ratio: 1:20
   💾 Reduction: 39,924,055 → 13,734 (0.03%)

🔄 Processing: balanced_1_100 (1:100 ratio (100 legitimate per fraud))
   ✅ Completed in 0.28s
   📊 Result: 311 fraud + 32,656 legitimate = 32,967 total
   📈 Actual ratio: 1:105
   💾 Reduction: 39,924,055 → 32,967 (0.08%)

🔄 Processing: balanced_1_1000 (1:1000 ratio (1000 legitimate per fraud))
   ✅ Completed in 0.73s
   📊 Result: 311 fraud + 326,994 legitimate = 327,305 total
   📈 Actual ratio: 1:1051
   💾 Reduction: 39,924,055 → 327,305 (0.82%)

🔄 Processing: balanced_1_10000 (1:10000 ratio (10000 legitimate per fraud))
   ✅ Completed in 4.44s
   📊 Result: 654 fraud + 6,049,661 legitimate = 6,050,315 total
   📈 Actual ratio: 1:9250
   💾 Reduction: 39,924,055 → 6,050,315 (15.15%)

🔄 Processing: balanced_1_100000 

In [142]:
# Summary of all balanced datasets
print("📊 SUMMARY: All Balanced Datasets")
print("=" * 80)

summary_data = []

for strategy in sampling_strategies:
    table_name = f"combined_features_{strategy['name']}"
    
    stats_query = f"""
    SELECT 
        fraud_flag,
        count(*) as count,
        round(count(*) * 100.0 / sum(count(*)) OVER (), 2) as percentage
    FROM public.{table_name}_distributed
    GROUP BY fraud_flag
    ORDER BY fraud_flag
    """
    
    result = clickhouse_client.execute(stats_query)
    
    legit = next((r[1] for r in result if r[0] == 0), 0)
    fraud = next((r[1] for r in result if r[0] == 1), 0)
    total = legit + fraud
    
    summary_data.append({
        'name': strategy['name'],
        'fraud': fraud,
        'legit': legit,
        'total': total,
        'ratio': f"1:{int(legit/fraud) if fraud > 0 else 0}",
        'fraud_pct': (fraud/total)*100 if total > 0 else 0
    })

print(f"\n{'Dataset':<25} {'Fraud':>12} {'Legitimate':>12} {'Total':>12} {'Ratio':>10} {'Fraud %':>10}")
print("-" * 80)

for data in summary_data:
    print(f"{data['name']:<25} {data['fraud']:>12,} {data['legit']:>12,} {data['total']:>12,} {data['ratio']:>10} {data['fraud_pct']:>9.4f}%")

print("-" * 80)
print(f"{'ORIGINAL (unbalanced)':<25} {fraud_count:>12,} {legit_count:>12,} {total_records:>12,} {'1:'+str(int(legit_count/fraud_count)):>10} {(fraud_count/total_records)*100:>9.4f}%")
print("=" * 80)

print("\n💡 Recommendations:")
print("   • Use balanced_1_1 for maximum balance (best for learning fraud patterns)")
print("   • Use balanced_1_5 or balanced_1_10 for moderate balance")
print("   • Use balanced_1_20 to retain more data while reducing imbalance")
print("\n✅ Balanced datasets ready for model training!")
print("=" * 80)

📊 SUMMARY: All Balanced Datasets

Dataset                          Fraud   Legitimate        Total      Ratio    Fraud %
--------------------------------------------------------------------------------
balanced_1_20                      654       13,080       13,734       1:20    4.7619%
balanced_1_100                     654       65,400       66,054      1:100    0.9901%
balanced_1_1000                    654      654,000      654,654     1:1000    0.0999%
balanced_1_10000                   654    6,540,000    6,540,654    1:10000    0.0100%
balanced_1_100000                  654   39,923,401   39,924,055    1:61044    0.0016%
--------------------------------------------------------------------------------
ORIGINAL (unbalanced)              654   39,923,401   39,924,055    1:61044    0.0016%

💡 Recommendations:
   • Use balanced_1_1 for maximum balance (best for learning fraud patterns)
   • Use balanced_1_5 or balanced_1_10 for moderate balance
   • Use balanced_1_20 to retain more 

In [ ]:
# Sample data from balanced dataset (1:1 ratio)
print("🔍 Sample Data from Balanced Dataset (1:1 ratio)")
print("=" * 80)

sample_query = """
SELECT 
    trans_id,
    fraud_flag,
    trx_channel,
    trx_type, 
FROM public.combined_features_balanced_1_20_distributed
WHERE fraud_flag=1
LIMIT 5
UNION ALL
SELECT 
    trans_id,
    fraud_flag,
    trx_channel,
    trx_type
FROM public.combined_features_balanced_1_20_distributed
WHERE fraud_flag=0
LIMIT 5
"""

print("\n📄 Sample Records (5 fraud + 5 legitimate):")
print("-" * 80)

sample_data = clickhouse_client.execute(sample_query)

print(f"{'Trans ID':<20} {'Fraud':>6} {'Channel':<15} {'Type':<15} {'Amount':>12} {'User_Txns_7d':>14} {'Avg_Amt_7d':>12}")
print("-" * 80)

for row in sample_data:
    trans_id, fraud, channel, txn_type = row
    fraud_label = "🚨 YES" if fraud == 1 else "✅ NO"
    print(f"{trans_id[:18]:<20} {fraud_label:>6} {channel[:13]:<15} {txn_type[:13]:<15}")

print("=" * 80)
print("✅ Balanced dataset ready for export and model training!")
print("=" * 80)

🔍 Sample Data from Balanced Dataset (1:1 ratio)

📄 Sample Records (5 fraud + 5 legitimate):
--------------------------------------------------------------------------------
Trans ID              Fraud Channel         Type                  Amount   User_Txns_7d   Avg_Amt_7d
--------------------------------------------------------------------------------
81947950385            ✅ NO USSD_API        IBFT Outgoing  
81948228573            ✅ NO USSD_API        Transfer(C2C)  
81948349488            ✅ NO API             Transfer(B2C)  
81948378240            ✅ NO NEW_JC_APP      IBFT Outgoing  
81948379799            ✅ NO NEW_JC_APP      Utility Bills  
81948373980           🚨 YES NEW_JC_APP      Transfer(C2B)  
81949352682           🚨 YES Payment Gatew   Online Paymen  
81949541293           🚨 YES Payment Gatew   Online Paymen  
81950080928           🚨 YES Payment Gatew   Online Paymen  
81954617435           🚨 YES Payment Gatew   Online Paymen  
✅ Balanced dataset ready for export and model

## Step 6: Feature Selection & Analysis

Comprehensive feature analysis using multiple techniques:
1. **Variance Analysis** - Remove low-variance features
2. **Correlation Analysis** - Identify multicollinearity
3. **VIF (Variance Inflation Factor)** - Detect multicollinearity
4. **Feature Importance** - Random Forest & XGBoost
5. **SHAP Values** - Explain feature contributions
6. **LIME** - Local interpretable explanations

**Dataset:** Using `combined_features_balanced_1_1` (equal fraud/legitimate ratio)

In [46]:
# Load balanced dataset for feature analysis
print("📥 Loading Balanced Dataset for Feature Analysis")
print("=" * 80)

# Fetch data from the 1:1 balanced dataset
fetch_query = """
SELECT * 
FROM public.combined_features_balanced_1_1_distributed
"""

print("⏳ Fetching data from ClickHouse...")
import time
start_time = time.time()

# Fetch data
data_raw = clickhouse_client.execute(fetch_query)

# Get column names
columns_query = """
SELECT name 
FROM system.columns 
WHERE database = 'public' 
  AND table = 'combined_features_balanced_1_1_local'
ORDER BY position
"""
column_names = [row[0] for row in clickhouse_client.execute(columns_query)]

# Create pandas DataFrame
import pandas as pd
import numpy as np

df = pd.DataFrame(data_raw, columns=column_names)

elapsed = time.time() - start_time

print(f"✅ Data loaded in {elapsed:.2f}s")
print(f"\n📊 Dataset Info:")
print(f"   • Total records: {len(df):,}")
print(f"   • Total columns: {len(df.columns):,}")
print(f"   • Fraud cases: {(df['fraud_flag'] == 1).sum():,}")
print(f"   • Legitimate cases: {(df['fraud_flag'] == 0).sum():,}")
print(f"   • Memory usage: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

print("\n📋 Column Types:")
print(f"   • Numeric: {df.select_dtypes(include=[np.number]).shape[1]}")
print(f"   • Categorical: {df.select_dtypes(include=['object']).shape[1]}")
print(f"   • DateTime: {df.select_dtypes(include=['datetime64']).shape[1]}")

print("=" * 80)

📥 Loading Balanced Dataset for Feature Analysis
⏳ Fetching data from ClickHouse...
✅ Data loaded in 0.04s

📊 Dataset Info:
   • Total records: 278
   • Total columns: 112
   • Fraud cases: 139
   • Legitimate cases: 139
   • Memory usage: 0.69 MB

📋 Column Types:
   • Numeric: 74
   • Categorical: 32
   • DateTime: 6


In [47]:
# Prepare features for analysis
print("🔧 Preparing Features for Analysis")
print("=" * 80)

# Separate target and features
target = df['fraud_flag']

# Exclude non-feature columns
exclude_cols = [
    'fraud_flag',  # Target
    'trans_id', 'ac_from', 'ac_to',  # Identifiers
    'data_date', 'trans_initiate_time', 'cutoff_date',  # Dates
    'processing_timestamp', 'created_at',  # Metadata
]

# Select numeric features only for variance, correlation, VIF analysis
numeric_features = df.select_dtypes(include=[np.number]).columns.tolist()
numeric_features = [col for col in numeric_features if col not in exclude_cols]

# Categorical features
categorical_features = df.select_dtypes(include=['object']).columns.tolist()
categorical_features = [col for col in categorical_features if col not in exclude_cols]

# Datetime features
datetime_features = df.select_dtypes(include=['datetime64']).columns.tolist()

print(f"📊 Feature Summary:")
print(f"   • Total features: {len(numeric_features) + len(categorical_features)}")
print(f"   • Numeric features: {len(numeric_features)}")
print(f"   • Categorical features: {len(categorical_features)}")
print(f"   • DateTime features: {len(datetime_features)} (excluded from analysis)")

print(f"\n🔢 Numeric Features ({len(numeric_features)}):")
for i in range(0, len(numeric_features), 5):
    print(f"   {', '.join(numeric_features[i:i+5])}")

print(f"\n📝 Categorical Features ({len(categorical_features)}):")
for i in range(0, len(categorical_features), 3):
    print(f"   {', '.join(categorical_features[i:i+3])}")

# Create feature matrix (numeric only for now)
X_numeric = df[numeric_features].copy()

print(f"\n✅ Feature matrix prepared: {X_numeric.shape}")
print("=" * 80)

🔧 Preparing Features for Analysis
📊 Feature Summary:
   • Total features: 99
   • Numeric features: 73
   • Categorical features: 26
   • DateTime features: 6 (excluded from analysis)

🔢 Numeric Features (73):
   start_balance, end_balance, trx_amt, mbar_credit_dl_ml_yl, mbar_debit_dl_ml_yl
   mbar_year_of_birth, hour_of_day, day_of_week, is_weekend, is_night
   is_business_hours, is_unusual_hour, night_weekend_combo, start_balance_log, balance_change
   balance_change_pct, txn_txns_3d, txn_total_amount_3d, txn_avg_amount_3d, txn_max_amount_3d
   txn_min_amount_3d, txn_unique_recipients_3d, txn_unique_channels_3d, txn_unique_types_3d, txn_is_high_activity_3d
   txn_multi_channel_recent, txn_amount_deviation_from_avg, txn_night_txns_3d, txn_weekend_txns_3d, channel_new_jc_app
   channel_ussd, channel_ussd_api, channel_payment_gateway, channel_mobile_app, type_transfer_c2c
   type_transfer_c2b, type_bill_payment, type_mobile_load, user_total_txns_3d, user_total_amount_3d
   user_avg_amou

### 6.1 Variance Analysis

Remove features with low or zero variance (constant or near-constant features)

In [48]:
# 1. VARIANCE ANALYSIS
print("📊 1. VARIANCE ANALYSIS")
print("=" * 80)

from sklearn.preprocessing import StandardScaler

# Calculate variance for each feature
variances = X_numeric.var()
variances_sorted = variances.sort_values(ascending=True)

print(f"\n🔍 Variance Statistics:")
print(f"   • Min variance: {variances.min():.6f}")
print(f"   • Max variance: {variances.max():.2f}")
print(f"   • Mean variance: {variances.mean():.2f}")
print(f"   • Median variance: {variances.median():.2f}")

# Identify low variance features (threshold: 0.01)
variance_threshold = 0.01
low_variance_features = variances[variances < variance_threshold].index.tolist()

print(f"\n⚠️  Low Variance Features (variance < {variance_threshold}):")
if low_variance_features:
    for feat in low_variance_features:
        print(f"   • {feat}: {variances[feat]:.6f}")
else:
    print(f"   ✅ No low variance features found")

# Show top 10 lowest variance features
print(f"\n📉 Top 10 Lowest Variance Features:")
for i, (feat, var) in enumerate(variances_sorted.head(10).items(), 1):
    print(f"   {i:2d}. {feat:<45} {var:>12.6f}")

# Show top 10 highest variance features
print(f"\n📈 Top 10 Highest Variance Features:")
for i, (feat, var) in enumerate(variances_sorted.tail(10).iloc[::-1].items(), 1):
    print(f"   {i:2d}. {feat:<45} {var:>12.2f}")

# Create filtered dataset (remove low variance features)
features_after_variance = [f for f in numeric_features if f not in low_variance_features]
X_variance_filtered = X_numeric[features_after_variance].copy()

print(f"\n✅ Variance Analysis Complete:")
print(f"   • Original features: {len(numeric_features)}")
print(f"   • Low variance features removed: {len(low_variance_features)}")
print(f"   • Remaining features: {len(features_after_variance)}")
print("=" * 80)

📊 1. VARIANCE ANALYSIS

🔍 Variance Statistics:
   • Min variance: 0.000000
   • Max variance: 5906852731416193483333825462272.00
   • Mean variance: 105779145104940788177147265024.00
   • Median variance: 19609769.59

⚠️  Low Variance Features (variance < 0.01):
   • mbar_credit_dl_ml_yl: 0.000000
   • mbar_debit_dl_ml_yl: 0.000000
   • day_of_week: 0.000000
   • is_weekend: 0.000000
   • txn_weekend_txns_3d: 0.000000
   • type_bill_payment: 0.000000
   • user_days_since_last_txn: 0.000000

📉 Top 10 Lowest Variance Features:
    1. mbar_credit_dl_ml_yl                              0.000000
    2. mbar_debit_dl_ml_yl                               0.000000
    3. day_of_week                                       0.000000
    4. is_weekend                                        0.000000
    5. txn_weekend_txns_3d                               0.000000
    6. type_bill_payment                                 0.000000
    7. user_days_since_last_txn                          0.000000
    8. 

### 6.2 Correlation Analysis

Identify highly correlated features (multicollinearity)

In [49]:
# 2. CORRELATION ANALYSIS
print("🔗 2. CORRELATION ANALYSIS")
print("=" * 80)

# Calculate correlation matrix
print("⏳ Calculating correlation matrix...")
correlation_matrix = X_variance_filtered.corr()

# Find highly correlated feature pairs
correlation_threshold = 0.90
high_corr_pairs = []

for i in range(len(correlation_matrix.columns)):
    for j in range(i+1, len(correlation_matrix.columns)):
        if abs(correlation_matrix.iloc[i, j]) >= correlation_threshold:
            high_corr_pairs.append({
                'feature1': correlation_matrix.columns[i],
                'feature2': correlation_matrix.columns[j],
                'correlation': correlation_matrix.iloc[i, j]
            })

print(f"\n🔍 High Correlation Pairs (|correlation| >= {correlation_threshold}):")
if high_corr_pairs:
    print(f"   Found {len(high_corr_pairs)} highly correlated pairs:")
    for i, pair in enumerate(high_corr_pairs[:20], 1):  # Show first 20
        print(f"   {i:2d}. {pair['feature1']:<40} ↔️ {pair['feature2']:<40} r={pair['correlation']:>6.3f}")
    if len(high_corr_pairs) > 20:
        print(f"   ... and {len(high_corr_pairs) - 20} more pairs")
else:
    print(f"   ✅ No highly correlated pairs found")

# Correlation with target
print(f"\n🎯 Correlation with Target (fraud_flag):")
target_corr = X_variance_filtered.corrwith(target).abs().sort_values(ascending=False)

print(f"\n📈 Top 15 Features Most Correlated with Fraud:")
for i, (feat, corr) in enumerate(target_corr.head(15).items(), 1):
    print(f"   {i:2d}. {feat:<45} r={corr:>6.4f}")

print(f"\n📉 Top 10 Features Least Correlated with Fraud:")
for i, (feat, corr) in enumerate(target_corr.tail(10).items(), 1):
    print(f"   {i:2d}. {feat:<45} r={corr:>6.4f}")

# Remove one feature from each highly correlated pair (keep the one more correlated with target)
features_to_remove = set()
for pair in high_corr_pairs:
    feat1, feat2 = pair['feature1'], pair['feature2']
    corr1 = abs(target_corr.get(feat1, 0))
    corr2 = abs(target_corr.get(feat2, 0))
    
    # Remove the feature less correlated with target
    if corr1 >= corr2:
        features_to_remove.add(feat2)
    else:
        features_to_remove.add(feat1)

features_after_correlation = [f for f in features_after_variance if f not in features_to_remove]
X_corr_filtered = X_variance_filtered[features_after_correlation].copy()

print(f"\n✅ Correlation Analysis Complete:")
print(f"   • Features before: {len(features_after_variance)}")
print(f"   • Highly correlated pairs: {len(high_corr_pairs)}")
print(f"   • Features removed: {len(features_to_remove)}")
print(f"   • Remaining features: {len(features_after_correlation)}")
print("=" * 80)

🔗 2. CORRELATION ANALYSIS
⏳ Calculating correlation matrix...

🔍 High Correlation Pairs (|correlation| >= 0.9):
   Found 195 highly correlated pairs:
    1. start_balance                            ↔️ end_balance                              r= 1.000
    2. start_balance                            ↔️ user_total_amount_3d                     r= 0.907
    3. start_balance                            ↔️ user_avg_amount_3d                       r= 0.952
    4. start_balance                            ↔️ user_median_amount_3d                    r= 0.933
    5. start_balance                            ↔️ user_total_amount_7d                     r= 0.906
    6. start_balance                            ↔️ user_avg_amount_7d                       r= 0.951
    7. start_balance                            ↔️ user_median_amount_7d                    r= 0.939
    8. start_balance                            ↔️ user_avg_start_balance_7d                r= 0.951
    9. start_balance                      

### 6.3 VIF (Variance Inflation Factor)

Detect multicollinearity using VIF (VIF > 10 indicates high multicollinearity)

In [50]:
# 3. VIF (Variance Inflation Factor) ANALYSIS
print("📐 3. VIF (Variance Inflation Factor) ANALYSIS")
print("=" * 80)

from statsmodels.stats.outliers_influence import variance_inflation_factor

# Handle missing values
X_vif = X_corr_filtered.fillna(X_corr_filtered.mean())

# Standardize features for VIF calculation
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_vif)
X_scaled_df = pd.DataFrame(X_scaled, columns=X_vif.columns)

print("⏳ Calculating VIF for all features...")
print("   (This may take a few minutes for many features...)")

# Calculate VIF for each feature
vif_data = []
for i, feature in enumerate(X_scaled_df.columns):
    if i % 10 == 0:
        print(f"   Progress: {i}/{len(X_scaled_df.columns)} features processed...")
    
    try:
        vif = variance_inflation_factor(X_scaled_df.values, i)
        vif_data.append({'feature': feature, 'VIF': vif})
    except:
        vif_data.append({'feature': feature, 'VIF': np.inf})

vif_df = pd.DataFrame(vif_data).sort_values('VIF', ascending=False)

print(f"\n✅ VIF calculation complete!")

print(f"\n🔍 VIF Statistics:")
vif_finite = vif_df[vif_df['VIF'] != np.inf]['VIF']
print(f"   • Min VIF: {vif_finite.min():.2f}")
print(f"   • Max VIF: {vif_finite.max():.2f}")
print(f"   • Mean VIF: {vif_finite.mean():.2f}")
print(f"   • Median VIF: {vif_finite.median():.2f}")

# Identify features with high VIF
vif_threshold = 10
high_vif_features = vif_df[vif_df['VIF'] > vif_threshold]['feature'].tolist()

print(f"\n⚠️  High VIF Features (VIF > {vif_threshold}):")
if high_vif_features:
    print(f"   Found {len(high_vif_features)} features with high VIF:")
    for i, row in enumerate(vif_df[vif_df['VIF'] > vif_threshold].head(20).itertuples(), 1):
        vif_val = "∞" if row.VIF == np.inf else f"{row.VIF:.2f}"
        print(f"   {i:2d}. {row.feature:<45} VIF={vif_val:>8}")
    if len(high_vif_features) > 20:
        print(f"   ... and {len(high_vif_features) - 20} more features")
else:
    print(f"   ✅ No high VIF features found")

print(f"\n📊 Top 10 Features with Lowest VIF (most independent):")
for i, row in enumerate(vif_df.tail(10).iloc[::-1].itertuples(), 1):
    print(f"   {i:2d}. {row.feature:<45} VIF={row.VIF:>8.2f}")

# Remove high VIF features
features_after_vif = [f for f in features_after_correlation if f not in high_vif_features]
X_vif_filtered = X_corr_filtered[features_after_vif].copy()

print(f"\n✅ VIF Analysis Complete:")
print(f"   • Features before: {len(features_after_correlation)}")
print(f"   • High VIF features removed: {len(high_vif_features)}")
print(f"   • Remaining features: {len(features_after_vif)}")
print("=" * 80)

📐 3. VIF (Variance Inflation Factor) ANALYSIS
⏳ Calculating VIF for all features...
   (This may take a few minutes for many features...)
   Progress: 0/33 features processed...
   Progress: 10/33 features processed...
   Progress: 20/33 features processed...
   Progress: 30/33 features processed...

✅ VIF calculation complete!

🔍 VIF Statistics:
   • Min VIF: 1.18
   • Max VIF: 33.63
   • Mean VIF: 6.26
   • Median VIF: 3.72

⚠️  High VIF Features (VIF > 10):
   Found 5 features with high VIF:
    1. user_max_amount_3d                            VIF=   33.63
    2. user_avg_amount_3d                            VIF=   33.60
    3. user_balance_volatility_7d                    VIF=   14.43
    4. txn_unique_types_3d                           VIF=   14.16
    5. txn_unique_channels_3d                        VIF=   13.35

📊 Top 10 Features with Lowest VIF (most independent):
    1. txn_amount_deviation_from_avg                 VIF=    1.18
    2. balance_change_pct                        

### 6.4 Feature Importance from Models

Train Random Forest and XGBoost to get feature importance scores

In [51]:
# 4. FEATURE IMPORTANCE FROM MODELS
print("🌲 4. FEATURE IMPORTANCE FROM MODELS")
print("=" * 80)

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_auc_score

# Prepare data
X_model = X_vif_filtered.fillna(X_vif_filtered.mean())
y_model = target

# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X_model, y_model, test_size=0.3, random_state=42, stratify=y_model
)

print(f"📊 Train/Test Split:")
print(f"   • Train: {len(X_train):,} samples ({(y_train == 1).sum()} fraud, {(y_train == 0).sum()} legit)")
print(f"   • Test:  {len(X_test):,} samples ({(y_test == 1).sum()} fraud, {(y_test == 0).sum()} legit)")

# === RANDOM FOREST ===
print(f"\n🌲 Training Random Forest Classifier...")
rf_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    min_samples_split=10,
    min_samples_leaf=5,
    random_state=42,
    n_jobs=-1,
    class_weight='balanced'
)

rf_model.fit(X_train, y_train)

# Predictions
y_pred_rf = rf_model.predict(X_test)
y_proba_rf = rf_model.predict_proba(X_test)[:, 1]

# Metrics
rf_auc = roc_auc_score(y_test, y_proba_rf)

print(f"✅ Random Forest trained!")
print(f"   • AUC-ROC: {rf_auc:.4f}")

print(f"\n📊 Classification Report (Random Forest):")
print(classification_report(y_test, y_pred_rf, target_names=['Legitimate', 'Fraud']))

# Feature importance
rf_importance = pd.DataFrame({
    'feature': X_model.columns,
    'importance': rf_model.feature_importances_
}).sort_values('importance', ascending=False)

print(f"\n🏆 Top 20 Most Important Features (Random Forest):")
for i, row in enumerate(rf_importance.head(20).itertuples(), 1):
    print(f"   {i:2d}. {row.feature:<45} {row.importance:>8.6f}")

print("=" * 80)

🌲 4. FEATURE IMPORTANCE FROM MODELS
📊 Train/Test Split:
   • Train: 194 samples (97 fraud, 97 legit)
   • Test:  84 samples (42 fraud, 42 legit)

🌲 Training Random Forest Classifier...
📊 Train/Test Split:
   • Train: 194 samples (97 fraud, 97 legit)
   • Test:  84 samples (42 fraud, 42 legit)

🌲 Training Random Forest Classifier...
✅ Random Forest trained!
   • AUC-ROC: 0.9121

📊 Classification Report (Random Forest):
              precision    recall  f1-score   support

  Legitimate       0.94      0.71      0.81        42
       Fraud       0.77      0.95      0.85        42

    accuracy                           0.83        84
   macro avg       0.85      0.83      0.83        84
weighted avg       0.85      0.83      0.83        84


🏆 Top 20 Most Important Features (Random Forest):
    1. trx_amt                                       0.150018
    2. balance_change                                0.142717
    3. channel_payment_gateway                       0.092625
    4. balance

In [52]:
# XGBoost Feature Importance
print("🚀 Training XGBoost Classifier...")
print("=" * 80)

import xgboost as xgb

# Train XGBoost
xgb_model = xgb.XGBClassifier(
    n_estimators=100,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1,
    scale_pos_weight=1  # Already balanced
)

xgb_model.fit(X_train, y_train)

# Predictions
y_pred_xgb = xgb_model.predict(X_test)
y_proba_xgb = xgb_model.predict_proba(X_test)[:, 1]

# Metrics
xgb_auc = roc_auc_score(y_test, y_proba_xgb)

print(f"✅ XGBoost trained!")
print(f"   • AUC-ROC: {xgb_auc:.4f}")

print(f"\n📊 Classification Report (XGBoost):")
print(classification_report(y_test, y_pred_xgb, target_names=['Legitimate', 'Fraud']))

# Feature importance (gain-based)
xgb_importance = pd.DataFrame({
    'feature': X_model.columns,
    'importance': xgb_model.feature_importances_
}).sort_values('importance', ascending=False)

print(f"\n🏆 Top 20 Most Important Features (XGBoost):")
for i, row in enumerate(xgb_importance.head(20).itertuples(), 1):
    print(f"   {i:2d}. {row.feature:<45} {row.importance:>8.6f}")

# Combined importance (average of RF and XGBoost)
combined_importance = pd.DataFrame({
    'feature': X_model.columns,
    'rf_importance': rf_importance.set_index('feature')['importance'],
    'xgb_importance': xgb_importance.set_index('feature')['importance']
})
combined_importance['avg_importance'] = (combined_importance['rf_importance'] + combined_importance['xgb_importance']) / 2
combined_importance = combined_importance.sort_values('avg_importance', ascending=False)

print(f"\n🎯 Top 20 Most Important Features (Combined RF + XGBoost):")
for i, row in enumerate(combined_importance.head(20).itertuples(), 1):
    print(f"   {i:2d}. {row.Index:<45} RF={row.rf_importance:.4f} XGB={row.xgb_importance:.4f} Avg={row.avg_importance:.4f}")

print("=" * 80)

🚀 Training XGBoost Classifier...
✅ XGBoost trained!
   • AUC-ROC: 0.9325

📊 Classification Report (XGBoost):
              precision    recall  f1-score   support

  Legitimate       0.88      0.69      0.77        42
       Fraud       0.75      0.90      0.82        42

    accuracy                           0.80        84
   macro avg       0.81      0.80      0.80        84
weighted avg       0.81      0.80      0.80        84


🏆 Top 20 Most Important Features (XGBoost):
    1. channel_payment_gateway                       0.114796
    2. channel_ussd_api                              0.097527
    3. txn_max_amount_3d                             0.093985
    4. type_transfer_c2c                             0.090517
    5. balance_change                                0.084700
    6. trx_amt                                       0.055595
    7. type_transfer_c2b                             0.046406
    8. channel_new_jc_app                            0.044963
    9. txn_min_amount_3

### 6.5 SHAP (SHapley Additive exPlanations)

Global and local feature importance using SHAP values

In [53]:
# 5. SHAP ANALYSIS
print("🎯 5. SHAP (SHapley Additive exPlanations) ANALYSIS")
print("=" * 80)

import shap

# Use a sample for SHAP (SHAP can be slow on large datasets)
sample_size = min(100, len(X_test))
X_shap_sample = X_test.sample(n=sample_size, random_state=42)

print(f"📊 Using sample of {sample_size} instances for SHAP analysis")
print(f"   (SHAP computation can be slow on large datasets)")

# === SHAP for XGBoost (Tree SHAP - fast) ===
print(f"\n🚀 Computing SHAP values for XGBoost...")

# Create SHAP explainer
explainer_xgb = shap.TreeExplainer(xgb_model)
shap_values_xgb = explainer_xgb.shap_values(X_shap_sample)

print(f"✅ SHAP values computed!")

# Calculate mean absolute SHAP values for global importance
shap_importance = pd.DataFrame({
    'feature': X_model.columns,
    'mean_abs_shap': np.abs(shap_values_xgb).mean(axis=0)
}).sort_values('mean_abs_shap', ascending=False)

print(f"\n🏆 Top 20 Most Important Features (SHAP - Global):")
for i, row in enumerate(shap_importance.head(20).itertuples(), 1):
    print(f"   {i:2d}. {row.feature:<45} Mean |SHAP|={row.mean_abs_shap:>8.6f}")

# SHAP Summary Statistics
print(f"\n📊 SHAP Value Statistics:")
print(f"   • Total features analyzed: {len(shap_importance)}")
print(f"   • Mean SHAP value: {np.abs(shap_values_xgb).mean():.6f}")
print(f"   • Max SHAP value: {np.abs(shap_values_xgb).max():.6f}")
print(f"   • Min SHAP value: {np.abs(shap_values_xgb).min():.6f}")

# Compare SHAP with model importance
print(f"\n🔍 Comparison: SHAP vs XGBoost Feature Importance")
comparison = pd.merge(
    shap_importance.head(15),
    xgb_importance.head(15),
    on='feature',
    how='outer'
).fillna(0).sort_values('mean_abs_shap', ascending=False)

print(f"\n{'Feature':<45} {'SHAP':>12} {'XGB Importance':>15}")
print("-" * 80)
for row in comparison.head(15).itertuples():
    print(f"{row.feature:<45} {row.mean_abs_shap:>12.6f} {row.importance:>15.6f}")

print("=" * 80)
print("✅ SHAP analysis complete!")
print("   💡 Tip: SHAP values show actual contribution to predictions")
print("   💡 Tip: Feature importance shows how often features are used")
print("=" * 80)

🎯 5. SHAP (SHapley Additive exPlanations) ANALYSIS


/root/miniconda3/envs/fraud/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


📊 Using sample of 84 instances for SHAP analysis
   (SHAP computation can be slow on large datasets)

🚀 Computing SHAP values for XGBoost...
✅ SHAP values computed!

🏆 Top 20 Most Important Features (SHAP - Global):
    1. channel_payment_gateway                       Mean |SHAP|=1.122073
    2. balance_change                                Mean |SHAP|=0.951870
    3. trx_amt                                       Mean |SHAP|=0.709405
    4. start_balance_log                             Mean |SHAP|=0.462679
    5. channel_new_jc_app                            Mean |SHAP|=0.398467
    6. user_unique_channels_7d                       Mean |SHAP|=0.367235
    7. is_business_hours                             Mean |SHAP|=0.306813
    8. user_type_diversity_score_7d                  Mean |SHAP|=0.254413
    9. balance_change_pct                            Mean |SHAP|=0.214952
   10. txn_max_amount_3d                             Mean |SHAP|=0.213771
   11. user_avg_time_between_txns_7d        

### 6.6 LIME (Local Interpretable Model-agnostic Explanations)

Explain individual predictions using LIME

In [57]:
# 6. LIME ANALYSIS
print("🔍 6. LIME (Local Interpretable Model-agnostic Explanations)")
print("=" * 80)

import lime
import lime.lime_tabular

# Create LIME explainer
lime_explainer = lime.lime_tabular.LimeTabularExplainer(
    training_data=X_train.values,
    feature_names=X_train.columns.tolist(),
    class_names=['Legitimate', 'Fraud'],
    mode='classification',
    random_state=42
)

print(f"✅ LIME explainer created")

# Select examples to explain
# Get one fraud and one legitimate example
fraud_idx = y_test[y_test == 1].index[0]
legit_idx = y_test[y_test == 0].index[0]

fraud_instance = X_test.loc[fraud_idx].values
legit_instance = X_test.loc[legit_idx].values

print(f"\n🔍 Explaining Individual Predictions:")

# === Explain FRAUD case ===
print(f"\n" + "="*80)
print(f"🚨 FRAUD CASE EXPLANATION")
print(f"="*80)

fraud_explanation = lime_explainer.explain_instance(
    fraud_instance,
    xgb_model.predict_proba,
    num_features=15
)

fraud_pred = xgb_model.predict_proba([fraud_instance])[0]
print(f"\n📊 Prediction Probabilities:")
print(f"   • Legitimate: {fraud_pred[0]:.4f}")
print(f"   • Fraud:      {fraud_pred[1]:.4f}")

print(f"\n🎯 Top 15 Features Contributing to FRAUD Prediction:")
fraud_features = fraud_explanation.as_list()
for i, (feature_desc, weight) in enumerate(fraud_features, 1):
    sign = "+" if weight > 0 else ""
    print(f"   {i:2d}. {feature_desc:<50} {sign}{weight:>8.4f}")

# === Explain LEGITIMATE case ===
print(f"\n" + "="*80)
print(f"✅ LEGITIMATE CASE EXPLANATION")
print(f"="*80)

legit_explanation = lime_explainer.explain_instance(
    legit_instance,
    xgb_model.predict_proba,
    num_features=15
)

legit_pred = xgb_model.predict_proba([legit_instance])[0]
print(f"\n📊 Prediction Probabilities:")
print(f"   • Legitimate: {legit_pred[0]:.4f}")
print(f"   • Fraud:      {legit_pred[1]:.4f}")

print(f"\n🎯 Top 15 Features Contributing to LEGITIMATE Prediction:")
legit_features = legit_explanation.as_list()
for i, (feature_desc, weight) in enumerate(legit_features, 1):
    sign = "+" if weight > 0 else ""
    print(f"   {i:2d}. {feature_desc:<50} {sign}{weight:>8.4f}")

print(f"\n" + "="*80)
print(f"✅ LIME analysis complete!")
print(f"   💡 Positive weights push towards Fraud (class 1)")
print(f"   💡 Negative weights push towards Legitimate (class 0)")
print("=" * 80)

🔍 6. LIME (Local Interpretable Model-agnostic Explanations)
✅ LIME explainer created

🔍 Explaining Individual Predictions:

🚨 FRAUD CASE EXPLANATION

📊 Prediction Probabilities:
   • Legitimate: 0.0131
   • Fraud:      0.9869

🎯 Top 15 Features Contributing to FRAUD Prediction:
    1. 0.00 < channel_payment_gateway <= 1.00             +  0.4300
    2. balance_change <= -5000.75                         +  0.2142
    3. user_unique_channels_7d <= 1.00                    +  0.1441
    4. trx_amt > 8000.00                                  +  0.1236
    5. channel_new_jc_app <= 0.00                          -0.1163
    6. is_business_hours <= 0.00                           -0.0938
    7. start_balance_log > 9.59                           +  0.0823
    8. channel_ussd_api <= 0.00                           +  0.0694
    9. user_avg_time_between_txns_7d > 23.25               -0.0678
   10. 0.60 < user_type_diversity_score_7d <= 0.80        +  0.0609
   11. txn_max_amount_3d <= 0.00            

### 6.7 Feature Selection Summary

Consolidate all feature selection results and create final feature set

In [58]:
# FEATURE SELECTION SUMMARY
print("📋 COMPREHENSIVE FEATURE SELECTION SUMMARY")
print("=" * 80)

# Create summary table
summary = pd.DataFrame({
    'Stage': [
        'Original Features',
        'After Variance Filter',
        'After Correlation Filter',
        'After VIF Filter',
        'Final Feature Set'
    ],
    'Feature Count': [
        len(numeric_features),
        len(features_after_variance),
        len(features_after_correlation),
        len(features_after_vif),
        len(features_after_vif)
    ],
    'Features Removed': [
        0,
        len(low_variance_features),
        len(features_to_remove),
        len(high_vif_features),
        0
    ],
    'Removal Reason': [
        '-',
        'Low variance',
        'High correlation',
        'High VIF (multicollinearity)',
        '-'
    ]
})

print(f"\n📊 Feature Selection Pipeline:")
print(summary.to_string(index=False))

# Calculate total reduction
total_removed = len(numeric_features) - len(features_after_vif)
reduction_pct = (total_removed / len(numeric_features)) * 100

print(f"\n📉 Feature Reduction:")
print(f"   • Original: {len(numeric_features)} features")
print(f"   • Final: {len(features_after_vif)} features")
print(f"   • Removed: {total_removed} features ({reduction_pct:.1f}%)")

# Top features by different methods
print(f"\n🏆 TOP 10 FEATURES BY DIFFERENT METHODS:")
print("=" * 80)

# Get top 10 from each method
top_variance = variances.sort_values(ascending=False).head(10).index.tolist()
top_corr = target_corr.head(10).index.tolist()
top_rf = rf_importance.head(10)['feature'].tolist()
top_xgb = xgb_importance.head(10)['feature'].tolist()
top_shap = shap_importance.head(10)['feature'].tolist()

# Find features that appear in multiple top-10 lists
all_top_features = top_variance + top_corr + top_rf + top_xgb + top_shap
feature_counts = pd.Series(all_top_features).value_counts()

print(f"\n🎯 Features Appearing in Multiple Top-10 Lists:")
consensus_features = feature_counts[feature_counts >= 3].index.tolist()

if consensus_features:
    print(f"\n   Features appearing in 3+ methods:")
    for i, feat in enumerate(consensus_features, 1):
        count = feature_counts[feat]
        methods = []
        if feat in top_variance: methods.append("Variance")
        if feat in top_corr: methods.append("Correlation")
        if feat in top_rf: methods.append("RF")
        if feat in top_xgb: methods.append("XGB")
        if feat in top_shap: methods.append("SHAP")
        
        print(f"   {i:2d}. {feat:<45} ({count}/5 methods: {', '.join(methods)})")
else:
    print("   No features appeared in 3+ methods")

# Recommended features (consensus + high importance)
print(f"\n💡 RECOMMENDED FEATURE SET:")
print(f"   Based on consensus and importance scores:")

# Take top features from combined importance
recommended_features = combined_importance.head(30)['feature'].tolist()

print(f"\n   Top 30 Features for Model Training:")
for i, feat in enumerate(recommended_features, 1):
    rf_imp = rf_importance.set_index('feature').loc[feat, 'importance']
    xgb_imp = xgb_importance.set_index('feature').loc[feat, 'importance']
    avg_imp = (rf_imp + xgb_imp) / 2
    print(f"   {i:2d}. {feat:<45} Importance={avg_imp:.6f}")

print("\n" + "=" * 80)
print("✅ FEATURE ANALYSIS COMPLETE!")
print("=" * 80)
print(f"\n📝 Summary Statistics:")
print(f"   • Models Trained: Random Forest, XGBoost")
print(f"   • RF AUC-ROC: {rf_auc:.4f}")
print(f"   • XGBoost AUC-ROC: {xgb_auc:.4f}")
print(f"   • Features Analyzed: {len(numeric_features)}")
print(f"   • Final Features: {len(features_after_vif)}")
print(f"   • Recommended Features: {len(recommended_features)}")
print(f"\n💡 Next Steps:")
print(f"   1. Use recommended features for model training")
print(f"   2. Export feature-selected dataset")
print(f"   3. Train production models with optimal features")
print("=" * 80)

📋 COMPREHENSIVE FEATURE SELECTION SUMMARY

📊 Feature Selection Pipeline:
                   Stage  Feature Count  Features Removed               Removal Reason
       Original Features             73                 0                            -
   After Variance Filter             66                 7                 Low variance
After Correlation Filter             33                33             High correlation
        After VIF Filter             28                 5 High VIF (multicollinearity)
       Final Feature Set             28                 0                            -

📉 Feature Reduction:
   • Original: 73 features
   • Final: 28 features
   • Removed: 45 features (61.6%)

🏆 TOP 10 FEATURES BY DIFFERENT METHODS:

🎯 Features Appearing in Multiple Top-10 Lists:

   Features appearing in 3+ methods:
    1. channel_payment_gateway                       (4/5 methods: Correlation, RF, XGB, SHAP)
    2. txn_max_amount_3d                             (4/5 methods: Correlati

In [59]:
# Save feature selection results
print("💾 Saving Feature Selection Results")
print("=" * 80)

import json

# Create results dictionary
feature_selection_results = {
    'metadata': {
        'dataset': 'combined_features_balanced_1_1',
        'total_records': len(df),
        'fraud_cases': int((df['fraud_flag'] == 1).sum()),
        'legitimate_cases': int((df['fraud_flag'] == 0).sum()),
        'cutoff_date': CUTOFF_DATE
    },
    'feature_counts': {
        'original': len(numeric_features),
        'after_variance': len(features_after_variance),
        'after_correlation': len(features_after_correlation),
        'after_vif': len(features_after_vif),
        'recommended': len(recommended_features)
    },
    'removed_features': {
        'low_variance': low_variance_features,
        'high_correlation': list(features_to_remove),
        'high_vif': high_vif_features
    },
    'recommended_features': recommended_features,
    'feature_importance': {
        'random_forest': rf_importance.to_dict('records'),
        'xgboost': xgb_importance.to_dict('records'),
        'combined': combined_importance.to_dict('records'),
        'shap': shap_importance.to_dict('records')
    },
    'model_performance': {
        'random_forest_auc': float(rf_auc),
        'xgboost_auc': float(xgb_auc)
    },
    'thresholds': {
        'variance_threshold': variance_threshold,
        'correlation_threshold': correlation_threshold,
        'vif_threshold': vif_threshold
    }
}

# Save to JSON file
output_file = '/root/research-dir/dev/jazzcash-fraud-detection/analysis/feature_selection_results.json'

import os
os.makedirs(os.path.dirname(output_file), exist_ok=True)

with open(output_file, 'w') as f:
    json.dump(feature_selection_results, f, indent=2)

print(f"✅ Results saved to: {output_file}")

# Save recommended features list
features_file = '/root/research-dir/dev/jazzcash-fraud-detection/analysis/recommended_features.txt'
with open(features_file, 'w') as f:
    f.write("# Recommended Features for Fraud Detection Model\n")
    f.write(f"# Generated: {pd.Timestamp.now()}\n")
    f.write(f"# Total Features: {len(recommended_features)}\n\n")
    for i, feat in enumerate(recommended_features, 1):
        f.write(f"{i}. {feat}\n")

print(f"✅ Recommended features saved to: {features_file}")

# Save feature importance comparison
importance_file = '/root/research-dir/dev/jazzcash-fraud-detection/analysis/feature_importance_comparison.csv'
combined_importance.to_csv(importance_file, index=False)

print(f"✅ Feature importance comparison saved to: {importance_file}")

print(f"\n📊 Files Created:")
print(f"   1. {output_file}")
print(f"   2. {features_file}")
print(f"   3. {importance_file}")

print("\n" + "=" * 80)
print("✅ FEATURE SELECTION COMPLETE!")
print("=" * 80)

💾 Saving Feature Selection Results
✅ Results saved to: /root/research-dir/dev/jazzcash-fraud-detection/analysis/feature_selection_results.json
✅ Recommended features saved to: /root/research-dir/dev/jazzcash-fraud-detection/analysis/recommended_features.txt
✅ Feature importance comparison saved to: /root/research-dir/dev/jazzcash-fraud-detection/analysis/feature_importance_comparison.csv

📊 Files Created:
   1. /root/research-dir/dev/jazzcash-fraud-detection/analysis/feature_selection_results.json
   2. /root/research-dir/dev/jazzcash-fraud-detection/analysis/recommended_features.txt
   3. /root/research-dir/dev/jazzcash-fraud-detection/analysis/feature_importance_comparison.csv

✅ FEATURE SELECTION COMPLETE!


## Step 7: MLflow Experiment Tracking

Track all experiments using MLflow:
- **Sampling experiments** - Different class imbalance ratios
- **Feature selection** - Variance, Correlation, VIF results
- **Model training** - Random Forest, XGBoost with metrics
- **SHAP & LIME** - Interpretability metrics

In [60]:
# Install MLflow
print("📦 Installing MLflow...")
print("=" * 80)

import subprocess
import sys

try:
    import mlflow
    print("✅ MLflow already installed")
    print(f"   Version: {mlflow.__version__}")
except ImportError:
    print("⏳ Installing MLflow...")
    result = subprocess.run(
        [sys.executable, "-m", "pip", "install", "mlflow", "-q"],
        capture_output=True,
        text=True
    )
    
    if result.returncode == 0:
        print("✅ MLflow installed successfully!")
        import mlflow
        print(f"   Version: {mlflow.__version__}")
    else:
        print(f"❌ Installation failed: {result.stderr}")
        raise Exception("Failed to install MLflow")

print("\n" + "=" * 80)

📦 Installing MLflow...
⏳ Installing MLflow...
✅ MLflow installed successfully!
   Version: 3.5.1



In [61]:
# Setup MLflow Experiment
print("🔧 Setting up MLflow Experiment")
print("=" * 80)

import mlflow
import mlflow.sklearn
import mlflow.xgboost

# Set tracking URI (local file storage)
mlflow_tracking_uri = "file:///root/research-dir/dev/jazzcash-fraud-detection/mlruns"
mlflow.set_tracking_uri(mlflow_tracking_uri)

print(f"📍 Tracking URI: {mlflow_tracking_uri}")

# Create or get experiment
experiment_name = "fraud-detection-feature-engineering"
experiment = mlflow.get_experiment_by_name(experiment_name)

if experiment is None:
    experiment_id = mlflow.create_experiment(
        experiment_name,
        tags={
            "project": "jazzcash-fraud-detection",
            "dataset": "stixor_mbar_v",
            "cutoff_date": CUTOFF_DATE,
            "version": "1.0"
        }
    )
    print(f"✅ Created new experiment: {experiment_name}")
else:
    experiment_id = experiment.experiment_id
    print(f"✅ Using existing experiment: {experiment_name}")

print(f"   Experiment ID: {experiment_id}")

# Set experiment
mlflow.set_experiment(experiment_name)

print(f"\n📊 MLflow UI: Run 'mlflow ui --backend-store-uri {mlflow_tracking_uri}' to view experiments")
print("=" * 80)

🔧 Setting up MLflow Experiment
📍 Tracking URI: file:///root/research-dir/dev/jazzcash-fraud-detection/mlruns
✅ Created new experiment: fraud-detection-feature-engineering
   Experiment ID: 440293988637387331

📊 MLflow UI: Run 'mlflow ui --backend-store-uri file:///root/research-dir/dev/jazzcash-fraud-detection/mlruns' to view experiments


### 7.1 Log Sampling Experiments

Log all balanced dataset creation experiments

In [62]:
# Log Sampling Experiments to MLflow
print("📊 Logging Sampling Experiments to MLflow")
print("=" * 80)

for strategy in sampling_strategies:
    table_name = f"combined_features_{strategy['name']}"
    
    # Get statistics for this balanced dataset
    stats_query = f"""
    SELECT 
        fraud_flag,
        count(*) as count
    FROM public.{table_name}_distributed
    GROUP BY fraud_flag
    ORDER BY fraud_flag
    """
    
    result = clickhouse_client.execute(stats_query)
    
    legit = next((r[1] for r in result if r[0] == 0), 0)
    fraud = next((r[1] for r in result if r[0] == 1), 0)
    total = legit + fraud
    actual_ratio = legit / fraud if fraud > 0 else 0
    fraud_pct = (fraud / total * 100) if total > 0 else 0
    
    # Start MLflow run
    with mlflow.start_run(run_name=f"sampling_{strategy['name']}"):
        # Log parameters
        mlflow.log_param("sampling_strategy", strategy['name'])
        mlflow.log_param("target_ratio", f"1:{strategy['ratio']}")
        mlflow.log_param("sampling_method", "random_undersampling")
        mlflow.log_param("source_table", "combined_features_distributed")
        mlflow.log_param("target_table", table_name)
        mlflow.log_param("cutoff_date", CUTOFF_DATE)
        
        # Log metrics
        mlflow.log_metric("fraud_count", fraud)
        mlflow.log_metric("legitimate_count", legit)
        mlflow.log_metric("total_records", total)
        mlflow.log_metric("actual_ratio", actual_ratio)
        mlflow.log_metric("fraud_percentage", fraud_pct)
        mlflow.log_metric("class_balance_score", min(fraud, legit) / max(fraud, legit) if max(fraud, legit) > 0 else 0)
        
        # Log tags
        mlflow.set_tags({
            "stage": "sampling",
            "imbalance_technique": "undersampling",
            "dataset_type": "balanced"
        })
        
        print(f"✅ Logged: {strategy['name']} - {total:,} records (1:{actual_ratio:.0f})")

print(f"\n✅ Logged {len(sampling_strategies)} sampling experiments to MLflow")
print("=" * 80)

📊 Logging Sampling Experiments to MLflow
✅ Logged: balanced_1_1 - 278 records (1:1)
✅ Logged: balanced_1_5 - 834 records (1:5)
✅ Logged: balanced_1_10 - 1,529 records (1:10)
✅ Logged: balanced_1_20 - 2,919 records (1:20)

✅ Logged 4 sampling experiments to MLflow


### 7.2 Log Feature Selection Experiments

Log variance, correlation, and VIF analysis results

In [63]:
# Log Feature Selection Experiments to MLflow
print("🔍 Logging Feature Selection Experiments to MLflow")
print("=" * 80)

# === Variance Analysis Run ===
with mlflow.start_run(run_name="feature_selection_variance"):
    # Parameters
    mlflow.log_param("selection_method", "variance_threshold")
    mlflow.log_param("variance_threshold", variance_threshold)
    mlflow.log_param("dataset", "balanced_1_1")
    
    # Metrics
    mlflow.log_metric("original_features", len(numeric_features))
    mlflow.log_metric("low_variance_features", len(low_variance_features))
    mlflow.log_metric("features_after_variance", len(features_after_variance))
    mlflow.log_metric("variance_min", float(variances.min()))
    mlflow.log_metric("variance_max", float(variances.max()))
    mlflow.log_metric("variance_mean", float(variances.mean()))
    mlflow.log_metric("variance_median", float(variances.median()))
    
    # Save variance analysis as artifact
    variance_df = pd.DataFrame({
        'feature': variances.index,
        'variance': variances.values
    }).sort_values('variance', ascending=False)
    
    variance_file = "/tmp/variance_analysis.csv"
    variance_df.to_csv(variance_file, index=False)
    mlflow.log_artifact(variance_file, "feature_analysis")
    
    # Tags
    mlflow.set_tags({
        "stage": "feature_selection",
        "method": "variance",
        "filter_type": "statistical"
    })
    
    print("✅ Logged: Variance Analysis")

# === Correlation Analysis Run ===
with mlflow.start_run(run_name="feature_selection_correlation"):
    # Parameters
    mlflow.log_param("selection_method", "correlation_threshold")
    mlflow.log_param("correlation_threshold", correlation_threshold)
    mlflow.log_param("dataset", "balanced_1_1")
    
    # Metrics
    mlflow.log_metric("features_before", len(features_after_variance))
    mlflow.log_metric("high_corr_pairs", len(high_corr_pairs))
    mlflow.log_metric("features_removed", len(features_to_remove))
    mlflow.log_metric("features_after_correlation", len(features_after_correlation))
    
    # Target correlation stats
    mlflow.log_metric("max_target_correlation", float(target_corr.max()))
    mlflow.log_metric("min_target_correlation", float(target_corr.min()))
    mlflow.log_metric("mean_target_correlation", float(target_corr.mean()))
    
    # Save correlation analysis
    corr_file = "/tmp/correlation_analysis.csv"
    target_corr.to_csv(corr_file, header=['correlation'])
    mlflow.log_artifact(corr_file, "feature_analysis")
    
    # Save high correlation pairs
    if high_corr_pairs:
        pairs_df = pd.DataFrame(high_corr_pairs)
        pairs_file = "/tmp/high_correlation_pairs.csv"
        pairs_df.to_csv(pairs_file, index=False)
        mlflow.log_artifact(pairs_file, "feature_analysis")
    
    # Tags
    mlflow.set_tags({
        "stage": "feature_selection",
        "method": "correlation",
        "filter_type": "multicollinearity"
    })
    
    print("✅ Logged: Correlation Analysis")

# === VIF Analysis Run ===
with mlflow.start_run(run_name="feature_selection_vif"):
    # Parameters
    mlflow.log_param("selection_method", "vif_threshold")
    mlflow.log_param("vif_threshold", vif_threshold)
    mlflow.log_param("dataset", "balanced_1_1")
    
    # Metrics
    mlflow.log_metric("features_before", len(features_after_correlation))
    mlflow.log_metric("high_vif_features", len(high_vif_features))
    mlflow.log_metric("features_after_vif", len(features_after_vif))
    
    # VIF stats (excluding infinite values)
    vif_finite = vif_df[vif_df['VIF'] != np.inf]['VIF']
    mlflow.log_metric("vif_min", float(vif_finite.min()))
    mlflow.log_metric("vif_max", float(vif_finite.max()))
    mlflow.log_metric("vif_mean", float(vif_finite.mean()))
    mlflow.log_metric("vif_median", float(vif_finite.median()))
    
    # Save VIF analysis
    vif_file = "/tmp/vif_analysis.csv"
    vif_df.to_csv(vif_file, index=False)
    mlflow.log_artifact(vif_file, "feature_analysis")
    
    # Tags
    mlflow.set_tags({
        "stage": "feature_selection",
        "method": "vif",
        "filter_type": "multicollinearity"
    })
    
    print("✅ Logged: VIF Analysis")

# === Overall Feature Selection Summary ===
with mlflow.start_run(run_name="feature_selection_summary"):
    # Parameters
    mlflow.log_param("original_features", len(numeric_features))
    mlflow.log_param("final_features", len(features_after_vif))
    mlflow.log_param("variance_threshold", variance_threshold)
    mlflow.log_param("correlation_threshold", correlation_threshold)
    mlflow.log_param("vif_threshold", vif_threshold)
    
    # Metrics
    total_removed = len(numeric_features) - len(features_after_vif)
    reduction_pct = (total_removed / len(numeric_features)) * 100
    
    mlflow.log_metric("total_features_removed", total_removed)
    mlflow.log_metric("feature_reduction_pct", reduction_pct)
    mlflow.log_metric("features_after_variance", len(features_after_variance))
    mlflow.log_metric("features_after_correlation", len(features_after_correlation))
    mlflow.log_metric("features_after_vif", len(features_after_vif))
    
    # Save summary
    summary_file = "/tmp/feature_selection_summary.json"
    with open(summary_file, 'w') as f:
        json.dump({
            'original_features': len(numeric_features),
            'after_variance': len(features_after_variance),
            'after_correlation': len(features_after_correlation),
            'after_vif': len(features_after_vif),
            'total_removed': total_removed,
            'reduction_pct': reduction_pct
        }, f, indent=2)
    mlflow.log_artifact(summary_file, "summaries")
    
    # Tags
    mlflow.set_tags({
        "stage": "feature_selection",
        "method": "complete_pipeline",
        "filter_type": "comprehensive"
    })
    
    print("✅ Logged: Feature Selection Summary")

print(f"\n✅ Logged 4 feature selection experiments to MLflow")
print("=" * 80)

🔍 Logging Feature Selection Experiments to MLflow
✅ Logged: Variance Analysis
✅ Logged: Correlation Analysis
✅ Logged: VIF Analysis
✅ Logged: Feature Selection Summary

✅ Logged 4 feature selection experiments to MLflow


### 7.3 Log Model Training Experiments

Log Random Forest and XGBoost model training with full metrics and artifacts

In [64]:
# Log Random Forest Model Experiment
print("🌲 Logging Random Forest Model Experiment to MLflow")
print("=" * 80)

from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

with mlflow.start_run(run_name="random_forest_model"):
    # Log parameters
    mlflow.log_param("model_type", "RandomForestClassifier")
    mlflow.log_param("n_estimators", 100)
    mlflow.log_param("max_depth", 10)
    mlflow.log_param("min_samples_split", 10)
    mlflow.log_param("min_samples_leaf", 5)
    mlflow.log_param("class_weight", "balanced")
    mlflow.log_param("random_state", 42)
    mlflow.log_param("n_features", len(features_after_vif))
    mlflow.log_param("train_size", len(X_train))
    mlflow.log_param("test_size", len(X_test))
    
    # Calculate metrics
    accuracy = accuracy_score(y_test, y_pred_rf)
    precision = precision_score(y_test, y_pred_rf)
    recall = recall_score(y_test, y_pred_rf)
    f1 = f1_score(y_test, y_pred_rf)
    
    # Confusion matrix
    cm = confusion_matrix(y_test, y_pred_rf)
    tn, fp, fn, tp = cm.ravel()
    
    # Log metrics
    mlflow.log_metric("accuracy", accuracy)
    mlflow.log_metric("precision", precision)
    mlflow.log_metric("recall", recall)
    mlflow.log_metric("f1_score", f1)
    mlflow.log_metric("auc_roc", rf_auc)
    mlflow.log_metric("true_positives", int(tp))
    mlflow.log_metric("true_negatives", int(tn))
    mlflow.log_metric("false_positives", int(fp))
    mlflow.log_metric("false_negatives", int(fn))
    mlflow.log_metric("specificity", tn / (tn + fp) if (tn + fp) > 0 else 0)
    
    # Log model
    mlflow.sklearn.log_model(rf_model, "model")
    
    # Log feature importance
    importance_file = "/tmp/rf_feature_importance.csv"
    rf_importance.to_csv(importance_file, index=False)
    mlflow.log_artifact(importance_file, "feature_importance")
    
    # Create and log confusion matrix plot
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
                xticklabels=['Legit', 'Fraud'], 
                yticklabels=['Legit', 'Fraud'])
    plt.title('Random Forest - Confusion Matrix')
    plt.ylabel('True Label')
    plt.xlabel('Predicted Label')
    cm_file = "/tmp/rf_confusion_matrix.png"
    plt.savefig(cm_file, dpi=100, bbox_inches='tight')
    plt.close()
    mlflow.log_artifact(cm_file, "plots")
    
    # Tags
    mlflow.set_tags({
        "stage": "model_training",
        "model_family": "ensemble",
        "algorithm": "random_forest",
        "task": "binary_classification"
    })
    
    print(f"✅ Logged Random Forest Model")
    print(f"   • AUC-ROC: {rf_auc:.4f}")
    print(f"   • Accuracy: {accuracy:.4f}")
    print(f"   • Precision: {precision:.4f}")
    print(f"   • Recall: {recall:.4f}")
    print(f"   • F1-Score: {f1:.4f}")

print("=" * 80)

🌲 Logging Random Forest Model Experiment to MLflow


2025/10/27 15:42:19 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2025/10/27 15:42:21 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.
2025/10/27 15:42:21 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


✅ Logged Random Forest Model
   • AUC-ROC: 0.9121
   • Accuracy: 0.8333
   • Precision: 0.7692
   • Recall: 0.9524
   • F1-Score: 0.8511


In [65]:
# Log XGBoost Model Experiment
print("🚀 Logging XGBoost Model Experiment to MLflow")
print("=" * 80)

with mlflow.start_run(run_name="xgboost_model"):
    # Log parameters
    mlflow.log_param("model_type", "XGBClassifier")
    mlflow.log_param("n_estimators", 100)
    mlflow.log_param("max_depth", 6)
    mlflow.log_param("learning_rate", 0.1)
    mlflow.log_param("subsample", 0.8)
    mlflow.log_param("colsample_bytree", 0.8)
    mlflow.log_param("scale_pos_weight", 1)
    mlflow.log_param("random_state", 42)
    mlflow.log_param("n_features", len(features_after_vif))
    mlflow.log_param("train_size", len(X_train))
    mlflow.log_param("test_size", len(X_test))
    
    # Calculate metrics
    accuracy = accuracy_score(y_test, y_pred_xgb)
    precision = precision_score(y_test, y_pred_xgb)
    recall = recall_score(y_test, y_pred_xgb)
    f1 = f1_score(y_test, y_pred_xgb)
    
    # Confusion matrix
    cm = confusion_matrix(y_test, y_pred_xgb)
    tn, fp, fn, tp = cm.ravel()
    
    # Log metrics
    mlflow.log_metric("accuracy", accuracy)
    mlflow.log_metric("precision", precision)
    mlflow.log_metric("recall", recall)
    mlflow.log_metric("f1_score", f1)
    mlflow.log_metric("auc_roc", xgb_auc)
    mlflow.log_metric("true_positives", int(tp))
    mlflow.log_metric("true_negatives", int(tn))
    mlflow.log_metric("false_positives", int(fp))
    mlflow.log_metric("false_negatives", int(fn))
    mlflow.log_metric("specificity", tn / (tn + fp) if (tn + fp) > 0 else 0)
    
    # Log model
    mlflow.xgboost.log_model(xgb_model, "model")
    
    # Log feature importance
    importance_file = "/tmp/xgb_feature_importance.csv"
    xgb_importance.to_csv(importance_file, index=False)
    mlflow.log_artifact(importance_file, "feature_importance")
    
    # Create and log confusion matrix plot
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Greens',
                xticklabels=['Legit', 'Fraud'],
                yticklabels=['Legit', 'Fraud'])
    plt.title('XGBoost - Confusion Matrix')
    plt.ylabel('True Label')
    plt.xlabel('Predicted Label')
    cm_file = "/tmp/xgb_confusion_matrix.png"
    plt.savefig(cm_file, dpi=100, bbox_inches='tight')
    plt.close()
    mlflow.log_artifact(cm_file, "plots")
    
    # Tags
    mlflow.set_tags({
        "stage": "model_training",
        "model_family": "gradient_boosting",
        "algorithm": "xgboost",
        "task": "binary_classification"
    })
    
    print(f"✅ Logged XGBoost Model")
    print(f"   • AUC-ROC: {xgb_auc:.4f}")
    print(f"   • Accuracy: {accuracy:.4f}")
    print(f"   • Precision: {precision:.4f}")
    print(f"   • Recall: {recall:.4f}")
    print(f"   • F1-Score: {f1:.4f}")

print("=" * 80)

2025/10/27 15:42:21 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🚀 Logging XGBoost Model Experiment to MLflow


/root/miniconda3/envs/fraud/lib/python3.10/site-packages/xgboost/sklearn.py:1028: UserWarning: [15:42:21] WARNING: /workspace/src/c_api/c_api.cc:1427: Saving model in the UBJSON format as default.  You can use file extension: `json`, `ubj` or `deprecated` to choose between formats.
  self.get_booster().save_model(fname)
2025/10/27 15:42:23 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.
2025/10/27 15:42:23 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


✅ Logged XGBoost Model
   • AUC-ROC: 0.9325
   • Accuracy: 0.7976
   • Precision: 0.7451
   • Recall: 0.9048
   • F1-Score: 0.8172


### 7.4 Log SHAP & LIME Interpretability Experiments

Log explainability metrics and visualizations

In [66]:
# Log SHAP Analysis Experiment
print("🎯 Logging SHAP Analysis Experiment to MLflow")
print("=" * 80)

with mlflow.start_run(run_name="shap_analysis_xgboost"):
    # Log parameters
    mlflow.log_param("explainer_type", "TreeExplainer")
    mlflow.log_param("model", "XGBoost")
    mlflow.log_param("sample_size", sample_size)
    mlflow.log_param("n_features", len(features_after_vif))
    
    # Log metrics
    mlflow.log_metric("mean_abs_shap", float(np.abs(shap_values_xgb).mean()))
    mlflow.log_metric("max_abs_shap", float(np.abs(shap_values_xgb).max()))
    mlflow.log_metric("min_abs_shap", float(np.abs(shap_values_xgb).min()))
    
    # Top features by SHAP
    top_10_shap = shap_importance.head(10)
    for i, row in enumerate(top_10_shap.itertuples(), 1):
        mlflow.log_metric(f"shap_importance_rank_{i}", row.mean_abs_shap)
        mlflow.log_param(f"shap_feature_rank_{i}", row.feature)
    
    # Log SHAP importance
    shap_file = "/tmp/shap_importance.csv"
    shap_importance.to_csv(shap_file, index=False)
    mlflow.log_artifact(shap_file, "interpretability")
    
    # Create SHAP summary plot (if matplotlib available)
    try:
        plt.figure(figsize=(10, 8))
        shap.summary_plot(shap_values_xgb, X_shap_sample, plot_type="bar", show=False)
        shap_plot_file = "/tmp/shap_summary_plot.png"
        plt.savefig(shap_plot_file, dpi=100, bbox_inches='tight')
        plt.close()
        mlflow.log_artifact(shap_plot_file, "plots")
    except Exception as e:
        print(f"   ⚠️  Could not create SHAP plot: {e}")
    
    # Tags
    mlflow.set_tags({
        "stage": "interpretability",
        "method": "shap",
        "explainer": "tree",
        "task": "global_importance"
    })
    
    print(f"✅ Logged SHAP Analysis")
    print(f"   • Mean |SHAP|: {np.abs(shap_values_xgb).mean():.6f}")
    print(f"   • Top feature: {shap_importance.iloc[0]['feature']}")

print("=" * 80)

🎯 Logging SHAP Analysis Experiment to MLflow
✅ Logged SHAP Analysis
   • Mean |SHAP|: 0.225417
   • Top feature: channel_payment_gateway
✅ Logged SHAP Analysis
   • Mean |SHAP|: 0.225417
   • Top feature: channel_payment_gateway


In [67]:
# Log LIME Analysis Experiment
print("🔍 Logging LIME Analysis Experiment to MLflow")
print("=" * 80)

with mlflow.start_run(run_name="lime_analysis_xgboost"):
    # Log parameters
    mlflow.log_param("explainer_type", "LimeTabular")
    mlflow.log_param("model", "XGBoost")
    mlflow.log_param("mode", "classification")
    mlflow.log_param("n_features_explained", 15)
    mlflow.log_param("training_size", len(X_train))
    
    # Fraud case metrics
    mlflow.log_param("fraud_case_true_label", "fraud")
    mlflow.log_param("fraud_case_predicted_prob_fraud", float(fraud_pred[1]))
    mlflow.log_param("fraud_case_predicted_prob_legit", float(fraud_pred[0]))
    
    # Legitimate case metrics
    mlflow.log_param("legit_case_true_label", "legitimate")
    mlflow.log_param("legit_case_predicted_prob_fraud", float(legit_pred[1]))
    mlflow.log_param("legit_case_predicted_prob_legit", float(legit_pred[0]))
    
    # Log explanations
    fraud_exp_file = "/tmp/lime_fraud_explanation.txt"
    with open(fraud_exp_file, 'w') as f:
        f.write("FRAUD CASE EXPLANATION\n")
        f.write("=" * 50 + "\n")
        f.write(f"Predicted Probability - Fraud: {fraud_pred[1]:.4f}\n")
        f.write(f"Predicted Probability - Legit: {fraud_pred[0]:.4f}\n\n")
        f.write("Top Features:\n")
        for i, (feat_desc, weight) in enumerate(fraud_features, 1):
            f.write(f"{i:2d}. {feat_desc:<50} {weight:>8.4f}\n")
    mlflow.log_artifact(fraud_exp_file, "lime_explanations")
    
    legit_exp_file = "/tmp/lime_legit_explanation.txt"
    with open(legit_exp_file, 'w') as f:
        f.write("LEGITIMATE CASE EXPLANATION\n")
        f.write("=" * 50 + "\n")
        f.write(f"Predicted Probability - Fraud: {legit_pred[1]:.4f}\n")
        f.write(f"Predicted Probability - Legit: {legit_pred[0]:.4f}\n\n")
        f.write("Top Features:\n")
        for i, (feat_desc, weight) in enumerate(legit_features, 1):
            f.write(f"{i:2d}. {feat_desc:<50} {weight:>8.4f}\n")
    mlflow.log_artifact(legit_exp_file, "lime_explanations")
    
    # Save LIME explanations as HTML
    try:
        fraud_html = fraud_explanation.as_html()
        fraud_html_file = "/tmp/lime_fraud_explanation.html"
        with open(fraud_html_file, 'w') as f:
            f.write(fraud_html)
        mlflow.log_artifact(fraud_html_file, "lime_explanations")
        
        legit_html = legit_explanation.as_html()
        legit_html_file = "/tmp/lime_legit_explanation.html"
        with open(legit_html_file, 'w') as f:
            f.write(legit_html)
        mlflow.log_artifact(legit_html_file, "lime_explanations")
    except Exception as e:
        print(f"   ⚠️  Could not save HTML explanations: {e}")
    
    # Tags
    mlflow.set_tags({
        "stage": "interpretability",
        "method": "lime",
        "explainer": "tabular",
        "task": "local_explanation"
    })
    
    print(f"✅ Logged LIME Analysis")
    print(f"   • Fraud case prediction: {fraud_pred[1]:.4f}")
    print(f"   • Legit case prediction: {legit_pred[0]:.4f}")

print("=" * 80)

🔍 Logging LIME Analysis Experiment to MLflow
✅ Logged LIME Analysis
   • Fraud case prediction: 0.9869
   • Legit case prediction: 0.9946


### 7.5 MLflow Experiment Summary

View all logged experiments and launch MLflow UI

In [68]:
# MLflow Experiment Summary
print("📊 MLFLOW EXPERIMENT TRACKING SUMMARY")
print("=" * 80)

# Get all runs from the experiment
experiment = mlflow.get_experiment_by_name(experiment_name)
runs = mlflow.search_runs(experiment_ids=[experiment.experiment_id])

print(f"\n🔬 Experiment: {experiment_name}")
print(f"   Experiment ID: {experiment.experiment_id}")
print(f"   Total Runs: {len(runs)}")
print(f"   Artifact Location: {experiment.artifact_location}")

# Group runs by stage
runs_by_stage = runs.groupby('tags.stage').size().to_dict() if 'tags.stage' in runs.columns else {}

print(f"\n📋 Runs by Stage:")
for stage, count in sorted(runs_by_stage.items()):
    print(f"   • {stage}: {count} runs")

# Show top runs by AUC-ROC (for models)
if 'metrics.auc_roc' in runs.columns:
    model_runs = runs[runs['metrics.auc_roc'].notna()].sort_values('metrics.auc_roc', ascending=False)
    
    print(f"\n🏆 Top Model Runs by AUC-ROC:")
    for i, row in enumerate(model_runs.head(5).itertuples(), 1):
        run_name = row._2 if hasattr(row, '_2') else 'unknown'  # run_name is typically second column
        auc = row._asdict().get('metrics.auc_roc', 0)
        print(f"   {i}. {run_name}: AUC-ROC = {auc:.4f}")

# Show sampling experiments
sampling_runs = runs[runs['tags.stage'] == 'sampling'] if 'tags.stage' in runs.columns else pd.DataFrame()

if len(sampling_runs) > 0:
    print(f"\n⚖️  Sampling Experiments:")
    for i, row in enumerate(sampling_runs.itertuples(), 1):
        run_dict = row._asdict()
        strategy = run_dict.get('params.sampling_strategy', 'unknown')
        total = run_dict.get('metrics.total_records', 0)
        ratio = run_dict.get('params.target_ratio', 'unknown')
        print(f"   {i}. {strategy}: {int(total):,} records ({ratio})")

# Feature selection summary
fs_runs = runs[runs['tags.stage'] == 'feature_selection'] if 'tags.stage' in runs.columns else pd.DataFrame()

if len(fs_runs) > 0:
    print(f"\n🔍 Feature Selection Experiments: {len(fs_runs)} runs")
    
# Interpretability runs
interp_runs = runs[runs['tags.stage'] == 'interpretability'] if 'tags.stage' in runs.columns else pd.DataFrame()

if len(interp_runs) > 0:
    print(f"\n🎯 Interpretability Experiments: {len(interp_runs)} runs")

print(f"\n" + "=" * 80)
print(f"📊 MLflow UI Commands:")
print(f"=" * 80)
print(f"\n💡 To view experiments in MLflow UI:")
print(f"   1. Open a terminal")
print(f"   2. Run: mlflow ui --backend-store-uri {mlflow_tracking_uri}")
print(f"   3. Open: http://localhost:5000")
print(f"\n💡 To compare runs:")
print(f"   • Select multiple runs in the UI")
print(f"   • Click 'Compare' to see side-by-side metrics")
print(f"\n💡 To view artifacts:")
print(f"   • Click on any run")
print(f"   • Navigate to 'Artifacts' tab")
print(f"   • View models, plots, and analysis files")

print(f"\n" + "=" * 80)
print(f"✅ ALL EXPERIMENTS LOGGED TO MLFLOW!")
print(f"=" * 80)

# Create summary report
summary_report = {
    'experiment_name': experiment_name,
    'experiment_id': experiment.experiment_id,
    'total_runs': len(runs),
    'runs_by_stage': runs_by_stage,
    'tracking_uri': mlflow_tracking_uri,
    'timestamp': pd.Timestamp.now().isoformat()
}

# Save summary
summary_report_file = '/root/research-dir/dev/jazzcash-fraud-detection/analysis/mlflow_experiment_summary.json'
with open(summary_report_file, 'w') as f:
    json.dump(summary_report, f, indent=2)

print(f"\n📄 Summary saved to: {summary_report_file}")
print("=" * 80)

📊 MLFLOW EXPERIMENT TRACKING SUMMARY

🔬 Experiment: fraud-detection-feature-engineering
   Experiment ID: 440293988637387331
   Total Runs: 12
   Artifact Location: file:///root/research-dir/dev/jazzcash-fraud-detection/mlruns/440293988637387331

📋 Runs by Stage:
   • feature_selection: 4 runs
   • interpretability: 2 runs
   • model_training: 2 runs
   • sampling: 4 runs

🏆 Top Model Runs by AUC-ROC:
   1. unknown: AUC-ROC = 0.0000
   2. unknown: AUC-ROC = 0.0000

⚖️  Sampling Experiments:
   1. unknown: 0 records (unknown)
   2. unknown: 0 records (unknown)
   3. unknown: 0 records (unknown)
   4. unknown: 0 records (unknown)

🔍 Feature Selection Experiments: 4 runs

🎯 Interpretability Experiments: 2 runs

📊 MLflow UI Commands:

💡 To view experiments in MLflow UI:
   1. Open a terminal
   2. Run: mlflow ui --backend-store-uri file:///root/research-dir/dev/jazzcash-fraud-detection/mlruns
   3. Open: http://localhost:5000

💡 To compare runs:
   • Select multiple runs in the UI
   • Cli

In [69]:
# Quick launch MLflow UI (optional - run in background)
print("🚀 Quick MLflow UI Launcher")
print("=" * 80)

print("\n💡 Option 1: Launch MLflow UI in terminal")
print(f"   Run this command in a separate terminal:")
print(f"   $ mlflow ui --backend-store-uri {mlflow_tracking_uri} --port 5000")

print("\n💡 Option 2: Launch MLflow UI in background")
print("   Uncomment and run the code below:")
print()

# Uncomment to launch MLflow UI automatically
# import subprocess
# import os
# 
# mlflow_process = subprocess.Popen(
#     ['mlflow', 'ui', '--backend-store-uri', mlflow_tracking_uri, '--port', '5000'],
#     stdout=subprocess.PIPE,
#     stderr=subprocess.PIPE,
#     cwd=os.path.dirname(mlflow_tracking_uri.replace('file://', ''))
# )
# 
# print(f"✅ MLflow UI launched in background (PID: {mlflow_process.pid})")
# print(f"   Access at: http://localhost:5000")
# print(f"   To stop: kill {mlflow_process.pid}")

print("\n📊 What you can do in MLflow UI:")
print("   ✓ Compare model performance across experiments")
print("   ✓ View feature importance rankings")
print("   ✓ Analyze sampling strategy effectiveness")
print("   ✓ Download trained models and artifacts")
print("   ✓ Track experiment parameters and metrics over time")
print("   ✓ Visualize confusion matrices and SHAP plots")

print("\n" + "=" * 80)
print("✅ MLFLOW EXPERIMENT TRACKING SETUP COMPLETE!")
print("=" * 80)

🚀 Quick MLflow UI Launcher

💡 Option 1: Launch MLflow UI in terminal
   Run this command in a separate terminal:
   $ mlflow ui --backend-store-uri file:///root/research-dir/dev/jazzcash-fraud-detection/mlruns --port 5000

💡 Option 2: Launch MLflow UI in background
   Uncomment and run the code below:


📊 What you can do in MLflow UI:
   ✓ Compare model performance across experiments
   ✓ View feature importance rankings
   ✓ Analyze sampling strategy effectiveness
   ✓ Download trained models and artifacts
   ✓ Track experiment parameters and metrics over time
   ✓ Visualize confusion matrices and SHAP plots

✅ MLFLOW EXPERIMENT TRACKING SETUP COMPLETE!


### 7.6 Access MLflow UI from Local Laptop

**SSH Port Forwarding** to access MLflow UI running on server from your local browser

In [2]:
mlflow_tracking_uri = "file:///root/research-dir/dev/jazzcash-fraud-detection/mlruns"

# Instructions for accessing MLflow UI on your local laptop
print("🌐 HOW TO ACCESS MLFLOW UI ON YOUR LOCAL LAPTOP")
print("=" * 80)

print("\n📋 STEP-BY-STEP GUIDE:")
print("=" * 80)

# Get server info
import socket
server_hostname = socket.gethostname()
server_ip = socket.gethostbyname(server_hostname)

print(f"\n🖥️  SERVER INFORMATION:")
print(f"   • Hostname: {server_hostname}")
print(f"   • IP Address: {server_ip}")
print(f"   • MLflow Port: 5000")
print(f"   • Tracking URI: {mlflow_tracking_uri}")

print(f"\n" + "=" * 80)
print(f"📍 STEP 1: Start MLflow UI on the Server")
print(f"=" * 80)

print(f"\nOn the SERVER (where this notebook is running), open a terminal and run:")
print(f"\n   mlflow ui --backend-store-uri {mlflow_tracking_uri} --host 0.0.0.0 --port 5000")
print(f"\n   Note: Using --host 0.0.0.0 allows external connections")
print(f"\n   You should see:")
print(f"   [INFO] Starting gunicorn 20.1.0")
print(f"   [INFO] Listening at: http://0.0.0.0:5000")

print(f"\n" + "=" * 80)
print(f"📍 STEP 2: Set Up SSH Port Forwarding (on your LOCAL laptop)")
print(f"=" * 80)

print(f"\n🔧 METHOD 1: SSH Port Forwarding (Recommended)")
print(f"\nOn your LOCAL laptop terminal, run:")
print(f"\n   ssh -L 5000:localhost:5000 username@{server_ip}")
print(f"\n   Replace:")
print(f"   • 'username' with your SSH username")
print(f"   • '{server_ip}' with actual server IP/hostname")
print(f"\n   Example:")
print(f"   ssh -L 5000:localhost:5000 root@{server_ip}")

print(f"\n🔧 METHOD 2: VS Code Port Forwarding (If using VS Code)")
print(f"\n   1. Open VS Code connected to the server")
print(f"   2. Click 'PORTS' tab in the bottom panel")
print(f"   3. Click 'Forward a Port' (+ button)")
print(f"   4. Enter port: 5000")
print(f"   5. VS Code will automatically forward the port")

print(f"\n🔧 METHOD 3: Using SSH Config (Permanent Setup)")
print(f"\nOn your LOCAL laptop, edit ~/.ssh/config:")
print(f"\n   Host mlflow-server")
print(f"       HostName {server_ip}")
print(f"       User your-username")
print(f"       LocalForward 5000 localhost:5000")
print(f"\nThen connect with:")
print(f"   ssh mlflow-server")

print(f"\n" + "=" * 80)
print(f"📍 STEP 3: Access MLflow UI in Your Browser")
print(f"=" * 80)

print(f"\nOnce port forwarding is active, open your LOCAL browser:")
print(f"\n   🌐 http://localhost:5000")
print(f"\n   or")
print(f"\n   🌐 http://127.0.0.1:5000")

print(f"\n✅ You should now see the MLflow UI!")

print(f"\n" + "=" * 80)
print(f"🔍 TROUBLESHOOTING")
print(f"=" * 80)

print(f"\n❌ If you can't connect:")
print(f"\n1. Check MLflow is running on server:")
print(f"   ps aux | grep mlflow")
print(f"   netstat -tlnp | grep 5000")
print(f"\n2. Check SSH port forwarding is active (on local laptop):")
print(f"   netstat -an | grep 5000    (Windows/Linux)")
print(f"   lsof -i :5000               (Mac/Linux)")
print(f"\n3. Check firewall settings on server:")
print(f"   sudo ufw status")
print(f"   sudo firewall-cmd --list-all")
print(f"\n4. Try different port if 5000 is busy:")
print(f"   Server: mlflow ui --port 5001")
print(f"   Local: ssh -L 5001:localhost:5001 user@{server_ip}")

print(f"\n" + "=" * 80)
print(f"💡 QUICK REFERENCE")
print(f"=" * 80)

print(f"\n📝 Complete workflow:")
print(f"\n   SERVER TERMINAL:")
print(f"   $ mlflow ui --backend-store-uri {mlflow_tracking_uri} --host 0.0.0.0 --port 5000")
print(f"\n   LOCAL LAPTOP TERMINAL:")
print(f"   $ ssh -L 5000:localhost:5000 root@{server_ip}")
print(f"\n   LOCAL BROWSER:")
print(f"   🌐 http://localhost:5000")

print(f"\n" + "=" * 80)
print(f"🎯 ALTERNATIVE: Direct Access (If server has public IP)")
print(f"=" * 80)

print(f"\nIf your server has a public IP and firewall allows:")
print(f"\n   🌐 http://{server_ip}:5000")
print(f"\n   ⚠️  WARNING: Only use this in secure networks!")
print(f"   Consider using authentication or VPN for production.")

print(f"\n" + "=" * 80)
print(f"✅ SETUP COMPLETE!")
print(f"=" * 80)

🌐 HOW TO ACCESS MLFLOW UI ON YOUR LOCAL LAPTOP

📋 STEP-BY-STEP GUIDE:

🖥️  SERVER INFORMATION:
   • Hostname: dfs-ai-app2
   • IP Address: 10.205.161.118
   • MLflow Port: 5000
   • Tracking URI: file:///root/research-dir/dev/jazzcash-fraud-detection/mlruns

📍 STEP 1: Start MLflow UI on the Server

On the SERVER (where this notebook is running), open a terminal and run:

   mlflow ui --backend-store-uri file:///root/research-dir/dev/jazzcash-fraud-detection/mlruns --host 0.0.0.0 --port 5000

   Note: Using --host 0.0.0.0 allows external connections

   You should see:
   [INFO] Starting gunicorn 20.1.0
   [INFO] Listening at: http://0.0.0.0:5000

📍 STEP 2: Set Up SSH Port Forwarding (on your LOCAL laptop)

🔧 METHOD 1: SSH Port Forwarding (Recommended)

On your LOCAL laptop terminal, run:

   ssh -L 5000:localhost:5000 username@10.205.161.118

   Replace:
   • 'username' with your SSH username
   • '10.205.161.118' with actual server IP/hostname

   Example:
   ssh -L 5000:localhost:500

### 7.7 Advanced MLflow Configuration

Configure MLflow to allow external connections with proper security settings

In [3]:
# Advanced MLflow Configuration for External Access
print("🔧 ADVANCED MLFLOW CONFIGURATION")
print("=" * 80)

import socket

# Get server information
server_hostname = socket.gethostname()
try:
    server_ip = socket.gethostbyname(server_hostname)
except:
    server_ip = "YOUR_SERVER_IP"

print("\n📋 CONFIGURATION OPTIONS FOR EXTERNAL ACCESS")
print("=" * 80)

print("\n🔐 OPTION 1: Basic External Access (Development/Testing)")
print("-" * 80)

basic_command = f"""mlflow ui \\
    --backend-store-uri {mlflow_tracking_uri} \\
    --host 0.0.0.0 \\
    --port 5000"""

print(f"\nCommand:")
print(basic_command)

print(f"\nExplanation:")
print(f"  • --host 0.0.0.0        : Listen on all network interfaces")
print(f"  • --port 5000           : Use port 5000")
print(f"\nAccess from:")
print(f"  • Local:  http://localhost:5000")
print(f"  • Remote: http://{server_ip}:5000")

print("\n" + "=" * 80)
print("🔐 OPTION 2: Secure External Access with Allowed Hosts")
print("-" * 80)

# Get local laptop IP (you'll need to replace this)
print("\n⚠️  First, find your LOCAL laptop's IP address:")
print("   • Windows: ipconfig")
print("   • Mac/Linux: ifconfig or ip addr")
print("   • Example: 192.168.1.100")

local_ip_example = "10.43.200.149"  # Example IP

secure_command = f"""mlflow ui \\
    --backend-store-uri {mlflow_tracking_uri} \\
    --host 0.0.0.0 \\
    --port 5000 \\
    --gunicorn-opts "--access-logfile - --error-logfile -" """

print(f"\nCommand (with logging):")
print(secure_command)

print("\n" + "=" * 80)
print("🔐 OPTION 3: Full Security with CORS and Allowed Hosts")
print("-" * 80)

print("\nFor production environments with web-based access:")

full_secure_command = f"""mlflow server \\
    --backend-store-uri {mlflow_tracking_uri} \\
    --host 0.0.0.0 \\
    --port 5000 \\
    --gunicorn-opts "--timeout 120 --access-logfile - --error-logfile -" """

print(f"\nCommand:")
print(full_secure_command)

print(f"\n📝 Note: MLflow 2.0+ handles CORS automatically for common scenarios")

print("\n" + "=" * 80)
print("🔐 OPTION 4: Using Nginx Reverse Proxy (Production Recommended)")
print("-" * 80)

print("\nFor production, use Nginx as a reverse proxy with authentication:")

nginx_config = f"""# /etc/nginx/sites-available/mlflow
server {{
    listen 80;
    server_name mlflow.yourdomain.com;
    
    location / {{
        proxy_pass http://localhost:5000;
        proxy_set_header Host $host;
        proxy_set_header X-Real-IP $remote_addr;
        proxy_set_header X-Forwarded-For $proxy_add_x_forwarded_for;
        proxy_set_header X-Forwarded-Proto $scheme;
        
        # Optional: Add basic auth
        auth_basic "MLflow Access";
        auth_basic_user_file /etc/nginx/.htpasswd;
    }}
}}"""

print("\n1. Install Nginx:")
print("   sudo apt-get install nginx")

print("\n2. Create Nginx config:")
print(nginx_config)

print("\n3. Enable and restart:")
print("   sudo ln -s /etc/nginx/sites-available/mlflow /etc/nginx/sites-enabled/")
print("   sudo nginx -t")
print("   sudo systemctl restart nginx")

print("\n4. Start MLflow (only localhost):")
print(f"   mlflow ui --backend-store-uri {mlflow_tracking_uri} --host 127.0.0.1 --port 5000")

print("\n" + "=" * 80)
print("🔥 FIREWALL CONFIGURATION")
print("=" * 80)

print("\n📝 Allow MLflow port through firewall:")

print("\nUbuntu/Debian (UFW):")
print("   sudo ufw allow 5000/tcp")
print("   sudo ufw status")

print("\nCentOS/RHEL (firewalld):")
print("   sudo firewall-cmd --permanent --add-port=5000/tcp")
print("   sudo firewall-cmd --reload")

print("\nCheck if port is open:")
print(f"   sudo netstat -tlnp | grep 5000")
print(f"   sudo lsof -i :5000")

print("\n" + "=" * 80)
print("🚀 RECOMMENDED SETUP FOR YOUR USE CASE")
print("=" * 80)

print("\n✅ For accessing from your LOCAL laptop via SSH tunnel:")
print("   (No need for --host 0.0.0.0 or firewall changes)")

recommended_server = f"""# On SERVER:
mlflow ui --backend-store-uri {mlflow_tracking_uri} --port 5000

# On LOCAL laptop:
ssh -L 5000:localhost:5000 root@{server_ip}

# Open browser on LOCAL laptop:
http://localhost:5000"""

print(recommended_server)

print("\n✅ For direct access from network (Development only):")

recommended_direct = f"""# On SERVER:
mlflow ui \\
    --backend-store-uri {mlflow_tracking_uri} \\
    --host 0.0.0.0 \\
    --port 5000

# Allow firewall:
sudo ufw allow 5000/tcp

# Open browser from ANY device on network:
http://{server_ip}:5000"""

print(recommended_direct)

print("\n" + "=" * 80)
print("⚠️  SECURITY WARNINGS")
print("=" * 80)

print("\n❌ DON'T DO THIS in production:")
print("   • Using --host 0.0.0.0 without authentication")
print("   • Exposing MLflow directly to the internet")
print("   • Using HTTP instead of HTTPS for remote access")

print("\n✅ DO THIS instead:")
print("   • Use SSH tunneling for remote access")
print("   • Use Nginx reverse proxy with authentication")
print("   • Use HTTPS with SSL certificates")
print("   • Restrict firewall to specific IP addresses")
print("   • Use VPN for team access")

print("\n" + "=" * 80)
print("📚 QUICK REFERENCE COMMANDS")
print("=" * 80)

commands_reference = f"""
# 1. LOCAL ACCESS ONLY (Most Secure)
mlflow ui --backend-store-uri {mlflow_tracking_uri}
# Access: http://localhost:5000 (SSH tunnel from laptop)

# 2. NETWORK ACCESS (Development)
mlflow ui --backend-store-uri {mlflow_tracking_uri} --host 0.0.0.0 --port 5000
# Access: http://{server_ip}:5000 (Any device on network)

# 3. CHECK IF MLFLOW IS RUNNING
ps aux | grep mlflow
netstat -tlnp | grep 5000
lsof -i :5000

# 4. STOP MLFLOW
pkill -f "mlflow ui"

# 5. RUN IN BACKGROUND
nohup mlflow ui --backend-store-uri {mlflow_tracking_uri} --host 0.0.0.0 --port 5000 > mlflow.log 2>&1 &
# Check log: tail -f mlflow.log
"""

print(commands_reference)

print("=" * 80)
print("✅ CONFIGURATION GUIDE COMPLETE!")
print("=" * 80)

🔧 ADVANCED MLFLOW CONFIGURATION

📋 CONFIGURATION OPTIONS FOR EXTERNAL ACCESS

🔐 OPTION 1: Basic External Access (Development/Testing)
--------------------------------------------------------------------------------

Command:
mlflow ui \
    --backend-store-uri file:///root/research-dir/dev/jazzcash-fraud-detection/mlruns \
    --host 0.0.0.0 \
    --port 5000

Explanation:
  • --host 0.0.0.0        : Listen on all network interfaces
  • --port 5000           : Use port 5000

Access from:
  • Local:  http://localhost:5000
  • Remote: http://10.205.161.118:5000

🔐 OPTION 2: Secure External Access with Allowed Hosts
--------------------------------------------------------------------------------

⚠️  First, find your LOCAL laptop's IP address:
   • Windows: ipconfig
   • Mac/Linux: ifconfig or ip addr
   • Example: 192.168.1.100

Command (with logging):
mlflow ui \
    --backend-store-uri file:///root/research-dir/dev/jazzcash-fraud-detection/mlruns \
    --host 0.0.0.0 \
    --port 5000 

### 7.9 Stop MLflow Process

How to stop MLflow running on a specific port

In [4]:
import subprocess
import os

print("🛑 STOPPING MLFLOW PROCESS ON PORT 5001")
print("=" * 80)

# Method 1: Find and kill process by port
print("\n📍 Method 1: Find process using port 5001")
print("-" * 80)

try:
    # Find process ID using lsof
    result = subprocess.run(['lsof', '-ti', ':5001'], 
                          capture_output=True, text=True)
    
    if result.stdout.strip():
        pid = result.stdout.strip()
        print(f"✅ Found process: PID {pid}")
        print(f"\nKilling process {pid}...")
        
        # Kill the process
        kill_result = subprocess.run(['kill', '-9', pid], 
                                   capture_output=True, text=True)
        
        if kill_result.returncode == 0:
            print(f"✅ Successfully killed process {pid}")
        else:
            print(f"❌ Error killing process: {kill_result.stderr}")
    else:
        print("❌ No process found on port 5001")
        
except FileNotFoundError:
    print("⚠️  lsof not found, trying alternative method...")
    
    # Alternative: using netstat
    try:
        result = subprocess.run(['netstat', '-tlnp'], 
                              capture_output=True, text=True)
        
        for line in result.stdout.split('\n'):
            if ':5001' in line and 'LISTEN' in line:
                # Extract PID from netstat output
                parts = line.split()
                if len(parts) > 6:
                    pid_program = parts[6]
                    if '/' in pid_program:
                        pid = pid_program.split('/')[0]
                        print(f"✅ Found process: PID {pid}")
                        os.system(f'kill -9 {pid}')
                        print(f"✅ Killed process {pid}")
                        break
        else:
            print("❌ No process found on port 5001")
    except Exception as e:
        print(f"❌ Error: {e}")

print("\n" + "=" * 80)
print("📍 Method 2: Kill all MLflow processes")
print("-" * 80)

try:
    # Find all mlflow processes
    result = subprocess.run(['pgrep', '-f', 'mlflow'], 
                          capture_output=True, text=True)
    
    if result.stdout.strip():
        pids = result.stdout.strip().split('\n')
        print(f"✅ Found {len(pids)} MLflow process(es): {', '.join(pids)}")
        
        for pid in pids:
            print(f"Killing MLflow process {pid}...")
            subprocess.run(['kill', '-9', pid])
        
        print(f"✅ All MLflow processes killed")
    else:
        print("❌ No MLflow processes found")
        
except Exception as e:
    print(f"❌ Error: {e}")

print("\n" + "=" * 80)
print("🔍 VERIFY: Check if port 5001 is now free")
print("-" * 80)

try:
    result = subprocess.run(['lsof', '-i', ':5001'], 
                          capture_output=True, text=True)
    
    if result.stdout.strip():
        print("❌ Port 5001 is still in use:")
        print(result.stdout)
    else:
        print("✅ Port 5001 is now free!")
        
except:
    # Try netstat as alternative
    result = subprocess.run(['netstat', '-tlnp'], 
                          capture_output=True, text=True)
    
    if ':5001' in result.stdout:
        print("❌ Port 5001 is still in use:")
        for line in result.stdout.split('\n'):
            if ':5001' in line:
                print(line)
    else:
        print("✅ Port 5001 is now free!")

print("\n" + "=" * 80)
print("📚 MANUAL COMMANDS (if above doesn't work)")
print("=" * 80)

commands = """
# Option 1: Kill by port (using lsof)
sudo lsof -ti :5001 | xargs kill -9

# Option 2: Kill by port (using fuser)
sudo fuser -k 5001/tcp

# Option 3: Find and kill manually
lsof -i :5001
# Note the PID, then:
kill -9 <PID>

# Option 4: Kill all MLflow processes
pkill -9 -f mlflow

# Option 5: Check what's running on port
netstat -tlnp | grep 5001
lsof -i :5001
"""

print(commands)

print("=" * 80)
print("✅ PROCESS STOP COMMANDS COMPLETE!")
print("=" * 80)

🛑 STOPPING MLFLOW PROCESS ON PORT 5001

📍 Method 1: Find process using port 5001
--------------------------------------------------------------------------------
✅ Found process: PID 2173111
2173115
2173116
2173117
2173118

Killing process 2173111
2173115
2173116
2173117
2173118...
❌ Error killing process: kill: cannot find process "2173111
2173115
2173116
2173117
2173118"


📍 Method 2: Kill all MLflow processes
--------------------------------------------------------------------------------
✅ Found 3 MLflow process(es): 2170591, 2173030, 2173111
Killing MLflow process 2170591...
Killing MLflow process 2173030...
Killing MLflow process 2173111...
✅ All MLflow processes killed

🔍 VERIFY: Check if port 5001 is now free
--------------------------------------------------------------------------------
❌ Port 5001 is still in use:
COMMAND     PID USER   FD   TYPE  DEVICE SIZE/OFF NODE NAME
python  2173115 root    3u  IPv4 7381925      0t0  TCP *:commplex-link (LISTEN)
python  2173116 root   